## Театр роботов

**Выполнили** Харунжин Максим, Сухарев Александр, Лемещенко Мария, Цапко Нелли.

In [1]:
import base64
from datetime import datetime
from hashlib import sha256
from io import BytesIO
import json
import os
from pathlib import Path, PurePosixPath
import sys
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from IPython.core.magic import register_cell_magic
from IPython.display import display, Markdown, HTML

MODE = os.environ.get("THEATRE_MODE", "replay")       # "live" запускает местные LLM.
SCENARIO = os.environ.get("THEATRE_SCENARIO", "extra")   # Вторая пьеса — "lab404".
SEED = int(os.environ.get("THEATRE_SEED", "1211"))
TURNS = 4                                           # Ходов на действие, в финале всегда 4.
PLANNER = "gpt-oss:20b"
MODEL_A = "qwen2.5:7b"
MODEL_B = "hf.co/unsloth/gemma-3-4b-it-GGUF:Q4_K_M"
EDITOR = None                                       # Сюда можно назначить "gpt-oss:20b" для оценки дублей.
READING_MODEL = None                                # При "gpt-oss:20b" после игры будет ещё одна читка.
SYNTHESIZE = os.environ.get("THEATRE_SYNTHESIZE", "1" if MODE == "live" else "0") == "1"
OLLAMA = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
if "://" not in OLLAMA:
    OLLAMA = "http://" + OLLAMA
if MODE not in ("live", "replay") or SCENARIO not in ("extra", "lab404"):
    raise ValueError("Доступны режимы replay/live и пьесы extra/lab404.")

# Читаем собственное вложение ноутбука. Переименованный файл тоже можно открыть.
# Для ядра с другим рабочим каталогом задайте THEATRE_NOTEBOOK полным путём.
def attached_files():
    requested = os.environ.get("THEATRE_NOTEBOOK")
    files = [Path(requested)] if requested else [Path("LLM_Theatre.ipynb"), *sorted(Path.cwd().glob("*.ipynb"))]
    for file in dict.fromkeys(files):
        if file.is_file():
            document = json.loads(file.read_text(encoding="utf-8"))
            if document.get("metadata", {}).get("agent_theatre_submission") == 1:
                cell = next(c for c in document["cells"] if c["id"] == "materials")
                return cell["attachments"], cell["metadata"]["sha256"]
    raise FileNotFoundError("Откройте ноутбук из папки файла или укажите переменную THEATRE_NOTEBOOK.")

attachments, checksums = attached_files()
def attachment(name):
    raw = base64.b64decode(next(iter(attachments[name].values())))
    if sha256(raw).hexdigest() != checksums[name]:
        raise ValueError(f"Повреждено вложение {name}")
    return raw

# Собственная временная папка исключает импорт из рабочего репозитория.
workspace = TemporaryDirectory(prefix="agent-theatre-notebook-")
bundle = Path(workspace.name)
package = bundle / "theatre_notebook"
package.mkdir()
for module_name in list(sys.modules):
    if module_name == package.name or module_name.startswith(package.name + "."):
        del sys.modules[module_name]
sys.path.insert(0, str(bundle))
with ZipFile(BytesIO(attachment("materials.zip"))) as archive:
    for entry in archive.infolist():
        relative = PurePosixPath(entry.filename)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Некорректный адрес во вложении.")
        path = bundle.joinpath(*relative.parts)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(archive.read(entry))
for key in ("extra", "lab404"):
    (bundle / "examples" / "library" / key / "play.m4a").write_bytes(attachment(key + ".m4a"))
del attachments
examples = bundle / "examples"
media_manifest = json.loads((bundle / "media.json").read_text(encoding="utf-8"))

@register_cell_magic
def theatre_module(name, cell):
    """Ячейка содержит обычный модуль Python. Компилируем его и сохраняем для относительных импортов."""
    name = name.strip()
    if not name.isidentifier():
        raise ValueError("Ожидалось имя модуля Python.")
    path = package / (name + ".py")
    compile(cell, str(path), "exec")
    path.write_text(cell, encoding="utf-8")

print(f"Режим {MODE} · пьеса {SCENARIO} · Python {sys.version.split()[0]}")

Режим replay · пьеса extra · Python 3.13.15


## Карточки ролей и формат ответа

В карточке записаны личная ставка, речевая манера, few-shot-пример, формы рода и голос. Формы Pydantic задают отдельно намерение актёра, произносимую реплику, ремарку и решение. Морфологическая проверка возвращает актёру заметные ошибки согласования.

In [2]:
%%theatre_module __init__
"""Симулятор русскоязычной камерной пьесы."""


In [3]:
%%theatre_module scenario
from __future__ import annotations

import tomllib
from dataclasses import dataclass
from pathlib import Path

DATA = Path(__file__).with_name("data")
SCENARIOS = ("extra", "lab404")
SPEAKER_GENDER = {"aidar": "male", "eugene": "male", "baya": "female",
                  "kseniya": "female", "xenia": "female"}
NARRATOR = "baya"


@dataclass(frozen=True)
class VoiceRole:
    name: str
    gender: str
    speaker: str

    def __post_init__(self) -> None:
        if self.gender not in {"male", "female"} or SPEAKER_GENDER.get(self.speaker) != self.gender:
            raise ValueError(f"{self.name}: проверьте пол роли и голос {self.speaker!r} в cast.toml")

    @property
    def agreement(self) -> str:
        return "женский род" if self.gender == "female" else "мужской род"


@dataclass(frozen=True)
class Actor(VoiceRole):
    voice: str
    stake: str
    sample: str
    family: list[str]
    tactic: str


@dataclass(frozen=True)
class Scenario:
    key: str
    title: str
    source: str
    cast: dict[str, Actor]
    researchers: tuple[str, ...]
    setting: str
    question: str
    notice: str
    arc: tuple[str, ...]
    blocking: list[str]
    cues: list[list[str]]
    library_tasks: list[dict[str, str]]
    challenge: str
    unanimous: str
    nomination: str
    unresolved: str
    archive: Path


def load_cast() -> dict[str, Actor]:
    with (DATA / "cast.toml").open("rb") as file:
        return {key: Actor(**card) for key, card in tomllib.load(file).items()}


def load_scenario(key: str) -> Scenario:
    if key not in SCENARIOS:
        raise ValueError(f"Неизвестная постановка: {key}")
    path = DATA / key
    cast = load_cast()
    with (path / "scenario.toml").open("rb") as file:
        story = tomllib.load(file)
    story["cast"] = {name: cast[name] for name in story["cast"]}
    voices = [NARRATOR, *(actor.speaker for actor in story["cast"].values())]
    if len(voices) != len(set(voices)):
        raise ValueError(f"{key}: назначьте каждой роли и рассказчице отдельный голос")
    story["arc"] = tuple(story["arc"])
    story["researchers"] = tuple(story["researchers"])
    return Scenario(key=key, archive=path / "archive", **story)


In [4]:
%%theatre_module schema
import re
from dataclasses import dataclass
from typing import Annotated

from pydantic import AfterValidator, BaseModel, ConfigDict, Field, field_validator


def nonblank(value: str) -> str:
    if not value.strip():
        raise ValueError("нужен непустой текст")
    return value.strip()


Text = Annotated[str, AfterValidator(nonblank)]


class Form(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)


class ScenePlan(Form):
    title: Text
    opening: Text
    objective: Text

    @field_validator("opening")
    @classmethod
    def keep_exit_closed(cls, value: str) -> str:
        # Все opening исполняются до последнего голоса. Развязкой распоряжается resolve().
        motion = r"(?<!не )\b(?:открыва|отпира|распахива|приоткрыва|открыла|открыли|отворя)\w*"
        door = r"\bдвер[ьи]\b"
        if re.search(rf"{door}[^.!?]{{0,35}}{motion}|{motion}[^.!?]{{0,35}}{door}", value, re.I):
            raise ValueError("в начале каждой сцены дверь ещё закрыта, опиши свет или движение актёра")
        return value


class Plan(Form):
    scenes: list[ScenePlan]

    @field_validator("scenes")
    @classmethod
    def four_scenes(cls, scenes: list[ScenePlan]) -> list[ScenePlan]:
        if len(scenes) != 4:
            raise ValueError("пьеса должна состоять ровно из четырёх сцен")
        return scenes


class Lookup(Form):
    query: str


class ToolCall(Form):
    action: str
    query: str = Field(description="Ключевые слова для архива или поиска по репликам. Можно оставить пустым.")
    supporters: int = Field(ge=1, le=4, description="Число сторонников в мысленном эксперименте, иначе 1.")


class Intention(Form):
    intent: Text
    to: str
    source: str
    vote: str


class Review(Form):
    model_config = ConfigDict(populate_by_name=True)
    issue: str = Field(description="Категория одной существенной ошибки, либо пустая строка.")
    note: str = Field(alias="поправка", description="Короткая инструкция актёру на русском языке.")


class EditorialCheck(Form):
    issue: str
    excerpt: str = Field(description="Точные слова проверяемого актёра, пусто при ok.")
    anchor: str = Field(description="Дословный фрагмент вопроса или факта, с которым возникло противоречие.")
    note: str = Field(description="Одна поправка актёру, пусто при ok.")


def stage_text_issue(value: str) -> str | None:
    if value and (not re.search(r"[а-яё]", value, re.I)
                  or re.search(r"[\u3040-\u30ff\u3400-\u9fff]", value)):
        return "перепиши текст целиком на русском языке"
    if re.search(r"\.txt\b|\b(?:source|stage|vote|query|line|select|challenge)\s*:", value, re.I):
        return "служебные метки нельзя произносить в пьесе"
    return None


class Delivery(Form):
    model_config = ConfigDict(populate_by_name=True)
    line: Text = Field(alias="реплика", description="Прямая речь одного актёра, без имени перед ней.")
    stage: str = Field(alias="жест", description="Один видимый жест именно этого актёра. Можно оставить пустым.")

    @field_validator("line")
    @classmethod
    def stage_language(cls, value: str) -> str:
        if issue := stage_text_issue(value):
            raise ValueError(issue)
        if re.search(r"(?:^|\n)\s*(?:Гермиона|Чендлер|Тони|Барби|Профессор)(?: [А-ЯЁ][а-яё]+)?\s*:", value):
            raise ValueError("нужна только собственная прямая речь, без имени говорящего")
        return value

    @field_validator("stage")
    @classmethod
    def optional_direction(cls, value: str) -> str:
        # Посторонняя подпись модели не должна лишать актёра корректной реплики.
        return "" if stage_text_issue(value) else value


class Speech(Delivery):
    to: str
    source: str
    vote: str
    edited: bool = False
    quality_issues: list[str] = Field(default_factory=list)


@dataclass(frozen=True)
class Turn:
    actor: str
    scene: int
    speech: Speech
    index: int = 0


In [5]:
%%theatre_module language
"""Согласование русской речи с ролью и перевод сценических ремарок в третье лицо."""

import re
from collections.abc import Sequence
from functools import lru_cache

from pymorphy3 import MorphAnalyzer

from .scenario import Actor, VoiceRole


@lru_cache(maxsize=1)
def morphology() -> MorphAnalyzer:
    return MorphAnalyzer(lang="ru")


@lru_cache(maxsize=4096)
def parses(word: str):
    return morphology().parse(word)


QUOTED = re.compile(r'«[^»]*»|„[^“]*“|“[^”]*”|"[^"]*"')
PREDICATE = r"(?:\s+(?:не|бы|уже|тоже|сама?|тогда|просто|точно|верно|лично|сейчас|никогда|ещё))*\s+([а-яё]+)"


def wrong_gender(line: str, actor: Actor, partners: Sequence[Actor] = (),
                 addressee: Actor | None = None) -> bool:
    """Проверяем явные «я была» и «Тони, ты готова». У чужой цитаты собственный говорящий."""
    plain = QUOTED.sub(" ", line.casefold())
    subjects = [(r"\bя", actor)]
    named = False
    for partner in partners:
        names = "|".join(re.escape(s.casefold()) for s in [partner.name, *partner.name.split()])
        subjects.append((rf"\b(?:{names})\s*[,!]\s*ты", partner))
        if re.search(rf"(?:^|[,.!?…]\s*)(?:{names})\s*[,!?.]", plain):
            named = True
    if addressee and not named:
        subjects.append((r"\bты", addressee))
    for subject, role in subjects:
        expected = "femn" if role.gender == "female" else "masc"
        for match in re.finditer(subject + PREDICATE, plain):
            # «Я голос» — существительное. Согласуем только предикаты с различимым родом.
            genders = {p.tag.gender for p in parses(match[1]) if p.tag.number == "sing"
                       and (p.tag.POS == "VERB" and p.tag.tense == "past"
                            or p.tag.POS in {"ADJS", "PRTS"} and p.tag.gender in {"masc", "femn"})}
            if genders and expected not in genders:
                return True
    # Нулевой субъект распространён в устном ответе «Не уверена, что ...».
    # Вопросительные фразы могут быть обращением ко второму лицу.
    for sentence in re.split(r"(?<=[.!?…])\s+", plain):
        if "?" in sentence:
            continue
        implied = re.match(r"^(?:не )?(уверен[а]?|соглас(?:ен|на)|готов[а]?|рад[а]?)\b", sentence)
        if implied and (implied[1].endswith("а") != (actor.gender == "female")):
            return True
    return False


def stage_direction(stage: str, actor: VoiceRole) -> str:
    """В старых журналах встречается «Поправляю очки». Рассказчица описывает действие актёра."""
    def third_person(match: re.Match) -> str:
        word = match[0]
        if word.casefold() == "я":
            return actor.name.split()[0]
        best = parses(word.casefold())[0]
        if best.tag.POS == "VERB" and best.tag.person == "1per" and best.tag.number == "sing":
            form = best.inflect({"3per", "sing"})
            if form:
                return form.word.capitalize() if word[0].isupper() else form.word
        return word
    # Кавычки могут содержать указание на чужую прямую речь.
    parts = re.split(r'(«[^»]*»|"[^"]*")', stage)
    return "".join(part if i % 2 else re.sub(r"[А-Яа-яЁё]+", third_person, part)
                   for i, part in enumerate(parts))


## Вызовы моделей

Обе актёрские модели получают собственную карточку и услышанную часть разговора. Локальный транспорт обращается к Ollama. Ответ с полями проходит валидацию, а в художественной реплике отдельно извлекается жест. Здесь можно прочесть параметры генерации и правила повторной попытки.

In [6]:
%%theatre_module llm
"""Два транспорта с одним контрактом. Структуру ответов проверяем и после HTTP."""

from __future__ import annotations

import json
import os
import re
from typing import Protocol, TypeVar

import httpx
from pydantic import ValidationError

from .schema import Delivery, Form
from .language import parses

T = TypeVar("T", bound=Form)
DEFAULT_A = "qwen2.5:7b"
DEFAULT_B = "hf.co/unsloth/gemma-3-4b-it-GGUF:Q4_K_M"


class GenerationError(RuntimeError):
    pass


class Generator(Protocol):
    def ask(self, model: str, system: str, prompt: str, form: type[T], *,
            choices: dict[str, list[str]] | None = None, budget: int = 500) -> T: ...


def parse_spoken_take(raw: str) -> Delivery:
    raw = raw.strip()
    if raw.startswith("{"):
        return Delivery.model_validate_json(raw)
    gesture = []
    speech = []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        action = re.fullmatch(r"\*([^*]+)\*|\(([^()]+)\)", line)
        if action:
            gesture.append(action[1] or action[2])
        else:
            def move_aside(match: re.Match) -> str:
                fragment = (match[1] or match[2]).strip()
                words = re.findall(r"[А-Яа-яЁё]+", fragment)
                # Сохраняем смысловое ударение вроде «это *мой* выбор».
                if not words or not any(parses(word)[0].tag.POS in {"VERB", "GRND"}
                                        for word in words[:3]):
                    return match[0]
                gesture.append(fragment)
                return " "
            line = re.sub(r"\*([^*\n]+)\*|\(([^()\n]+)\)", move_aside, line)
            speech.append(line.removeprefix("—").strip())
    spoken = "\n".join(speech).strip()
    if spoken.startswith("«") and spoken.endswith("»"):
        spoken = spoken[1:-1]
    words = Delivery(line=spoken, stage=" ".join(gesture))
    return words.model_copy(update={"line": " ".join(words.line.split())})


class Gateway:
    def __init__(self, http: httpx.Client, ollama_url: str, seed: int):
        self.http = http
        self.ollama_url = ollama_url.rstrip("/")
        self.seed = seed

    def _post(self, url: str, payload: dict, headers: dict | None = None) -> dict:
        try:
            response = self.http.post(url, json=payload, headers=headers)
            response.raise_for_status()
            return response.json()
        except httpx.HTTPStatusError as exc:
            detail = "проверьте ключ и доступ к модели" if headers else exc.response.text[:400]
            if not headers and exc.response.status_code == 500 and "token repeat limit" in detail:
                # Ollama оборвала декодирование до получения JSON. Новый запрос с
                # короткой корректировкой проходит через тот же лимит, что и битый JSON.
                raise ValueError("генерация зациклилась, сформулируй короткий ответ другими словами") from exc
            raise GenerationError(f"{exc.response.status_code} от {url}: {detail}") from exc
        except (httpx.RequestError, ValueError) as exc:
            raise GenerationError(f"Не получен ответ от {url}: {exc}") from exc

    def _complete(self, model: str, system: str, prompt: str, schema: dict | None,
                  name: str, budget: int) -> str:
        if model.startswith("openai/"):
            key = os.getenv("OPENAI_API_KEY")
            if not key:
                raise GenerationError("Для выбранной модели openai/ нужен OPENAI_API_KEY.")
            body = self._post("https://api.openai.com/v1/responses", {
                "model": model.removeprefix("openai/"), "instructions": system,
                "input": prompt, "store": False, "max_output_tokens": max(2000, 3 * budget),
                "text": {"format": {"type": "json_schema", "name": name,
                                    "schema": schema, "strict": True}},
            }, {"Authorization": f"Bearer {key}"})
            if body.get("status") != "completed":
                raise ValueError(f"статус Responses API: {body.get('status')}")
            return "".join(part["text"] for item in body["output"] if item["type"] == "message"
                           for part in item["content"] if part["type"] == "output_text")
        # У GPT-OSS именованные уровни reasoning. В общий лимит Ollama входят
        # и эти токены, даже когда приложение читает только финальный JSON.
        is_gpt_oss = model.rsplit("/", 1)[-1].split(":")[0] == "gpt-oss"
        body = self._post(self.ollama_url + "/api/chat", {
            "model": model, "stream": False, "think": "low" if is_gpt_oss else False,
            **({"format": schema} if schema is not None else {}),
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": prompt}],
            "options": {"seed": self.seed, "temperature": 0.65, "num_ctx": 8192,
                        "num_predict": budget + (1500 if is_gpt_oss else 0)},
        })
        if body.get("done_reason") == "length":
            raise ValueError("исчерпан лимит генерации")
        return body["message"]["content"]

    def ask(self, model: str, system: str, prompt: str, form: type[T], *,
            choices: dict[str, list[str]] | None = None, budget: int = 500) -> T:
        schema = form.model_json_schema()
        choices = choices or {}
        for field, allowed in choices.items():
            schema["properties"][field]["enum"] = allowed
        spoken = form is Delivery and not model.startswith("openai/")
        if spoken:
            prompt += ("\nНапиши только слова персонажа без имени перед ними. Можно добавить один видимый "
                       "жест отдельной строкой в *звёздочках*. Закончи мысль. Пиши обычным текстом.")
        elif not model.startswith("openai/"):
            fields = ", ".join(schema["properties"])
            allowed = " ".join(f"{key} ∈ {json.dumps(values, ensure_ascii=False)}." for key, values in choices.items())
            prompt += f"\nВерни один объект {form.__name__} в JSON. Обязательные поля: {fields}. {allowed}"
        correction = ""
        for attempt in range(2):
            try:
                raw = self._complete(model, system, prompt + correction, None if spoken else schema,
                                     form.__name__, budget * (attempt + 1))
                result = parse_spoken_take(raw) if spoken else form.model_validate_json(raw)
                for field, allowed in choices.items():
                    if getattr(result, field) not in allowed:
                        raise ValueError(f"{field}: допустимы {json.dumps(allowed, ensure_ascii=False)}")
                return result
            except (ValidationError, ValueError, KeyError, TypeError) as exc:
                if attempt == 1:
                    raise GenerationError(f"{model}: некорректный ответ для {form.__name__}: {exc}") from exc
                detail = ("; ".join(f"{'.'.join(map(str, e['loc']))}: {e['msg']}" for e in exc.errors(include_input=False))
                          if isinstance(exc, ValidationError) else str(exc))
                output = "Дай одну законченную реплику по-русски." if spoken else "Верни объект JSON с текстом по-русски."
                correction = f"\nПерепиши ответ. {detail[:300]}. {output}"
        raise AssertionError("unreachable")


## Реквизит, память и чтение больших книг

Устройство описано в небольших текстовых файлах с номерами строк. Для классических пьес построен отдельный индекс русских лемм. Актёр может перечитать слова партнёра, проверить порог голосования или выбрать книжное свидетельство. Второй актёр получает доступ к соседним строкам только после того, как первый прочитал фрагмент вслух.

In [7]:
%%theatre_module archive
"""Поиск по реквизиту текущей пьесы. Модель передаёт слова, а не путь к файлу."""

import re
from dataclasses import dataclass
from pathlib import Path

STOP = set("как кто что это для или при есть был была какую какой один всех этих меня нас них наш чем если каких можно проверить найти узнать".split())


def terms(text: str) -> set[str]:
    # Префиксы сближают «кнопку» и «кнопка», сохраняя поиск прозрачным для отчёта.
    words = re.findall(r"[а-яa-z0-9]+", text.casefold().replace("ё", "е"))
    return {word[:5] for word in words if len(word) >= 3 and word not in STOP}


@dataclass(frozen=True)
class Hit:
    file: str
    line: int
    text: str

    @property
    def ref(self) -> str:
        return f"{self.file}:{self.line}"


class Archive:
    def __init__(self, directory: Path):
        self.entries: list[tuple[Hit, set[str]]] = []
        self.titles: list[str] = []
        for path in sorted(directory.glob("*.txt")):
            lines = path.read_text(encoding="utf-8").splitlines()
            if not lines:
                continue
            self.titles.append(lines[0])
            for number, text in enumerate(lines[1:], 2):
                if text.strip():
                    hit = Hit(path.name, number, text.strip())
                    self.entries.append((hit, terms(lines[0] + " " + text)))

    def search(self, query: str, limit: int = 2) -> list[Hit]:
        wanted = terms(query)
        ranked = [(len(wanted & words), hit) for hit, words in self.entries]
        ranked.sort(key=lambda item: -item[0])  # порядок файлов устойчив и при равном счёте
        return [hit for score, hit in ranked if score][:limit]


In [8]:
%%theatre_module library
"""Локальный поиск по TEI. Адрес результата указывает на строки неизменяемой редакции."""
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from functools import lru_cache
import hashlib
import json
import math
from pathlib import Path
import re
import unicodedata
from xml.etree.ElementTree import TreeBuilder
from xml.parsers import expat

from .language import parses
from .scenario import DATA

DIRECTORY = DATA / "library"
WORDS = re.compile(r"[а-яёa-z0-9]+", re.I)
STOP = set("быть сказать говорить спросить мочь хотеть найти показать пьеса книга произведение сцена "
           "эпизод место отрывок герой реплика слово вспомнить именно очень который какой такой этот".split())
GROUPS = ("жребий монета орлянка решка решётка орёл", "тайный инкогнито тайком секрет",
          "забыть оставить позабыть", "письмо конверт корреспонденция", "торги аукцион торг",
          "инспектор проверяющий ревизор", "запереть запирать замок", "больница лечебница госпиталь")


@lru_cache(maxsize=32768)
def lemma(word: str) -> str:
    p = parses(word)[0]
    if p.tag.POS in {"PREP", "CONJ", "PRCL", "NPRO", "INTJ"}:
        return ""
    return p.normal_form.replace("ё", "е")


def tokens(text: str) -> list[str]:
    text = unicodedata.normalize("NFC", text).casefold()
    return [term for word in WORDS.findall(text) if len(word) > 1
            if (term := lemma(word)) and term not in STOP]


def xml_with_lines(raw: bytes):
    # ElementTree из стандартной библиотеки не хранит номера строк. Expat отдаёт их
    # в момент чтения исходных байтов, включая вложенные авторские ремарки.
    builder = TreeBuilder()
    parser = expat.ParserCreate()
    spans = {}
    def opening(tag, attrs):
        node = builder.start(tag, attrs)
        spans[node] = [parser.CurrentLineNumber, 0]
    def closing(tag):
        node = builder.end(tag)
        spans[node][1] = parser.CurrentLineNumber
    parser.StartElementHandler = opening
    parser.EndElementHandler = closing
    parser.CharacterDataHandler = builder.data
    parser.Parse(raw, True)
    return builder.close(), spans


def clean(text: str) -> str:
    return unicodedata.normalize("NFC", " ".join(text.split()))


def chunks(text: str, size: int = 56):
    words = clean(text).split()
    while words:
        n = min(size, len(words))
        if n < len(words):
            n = next((i for i in range(n, n // 2, -1) if words[i-1].rstrip('»")').endswith((".", "!", "?", "…"))), n)
        yield " ".join(words[:n])
        words = words[n:]


@dataclass(frozen=True)
class Book:
    key: str
    author: str
    title: str
    file: str
    sha256: str


@dataclass(frozen=True)
class Passage:
    ref: str
    book: str
    act: int
    section: int
    heading: str
    speaker: str
    text: str
    first_line: int
    last_line: int


def passages(book: Book, raw: bytes):
    root, spans = xml_with_lines(raw)
    body = root.find("./text/body")
    if body is None:
        raise ValueError(f"{book.file}: отсутствует текст пьесы")
    serial = 0

    def parts(node, speaker):
        start, end = spans[node]
        if node.tag in {"p", "l"}:
            # В «Ревизоре» речь нередко прерывается ремаркой внутри того же абзаца.
            buf = node.text or ""
            for sub in node:
                if sub.tag == "stage":
                    yield speaker, buf, start, spans[sub][0]
                    yield "Ремарка", "".join(sub.itertext()), *spans[sub]
                    start, buf = spans[sub][1], ""
                else:
                    buf += "".join(sub.itertext())
                buf += sub.tail or ""
            yield speaker, buf, start, end
        elif node.tag == "stage":
            yield "Ремарка", "".join(node.itertext()), start, end
        elif node.tag in {"sp", "spGrp", "lg"}:
            if node.tag == "sp":
                speaker = clean(node.findtext("speaker") or node.get("who", "").replace("#", "")).rstrip(" .")
            for sub in node:
                yield from parts(sub, speaker)

    for act_no, act in enumerate(body.findall("div[@type='act']"), 1):
        section = 0
        for block in act:
            if block.tag == "div":
                section += 1
                head = clean(block.findtext("head") or f"Мизансцена {section}")
                nodes = list(block)
                section_no = section
            else:
                head, nodes, section_no = "Начало действия", [block], 0
            for node in nodes:
                for speaker, text, start, end in parts(node, ""):
                    for fragment in chunks(text):
                        serial += 1
                        yield Passage(f"{book.key}:{serial}", book.key, act_no, section_no, head,
                                      speaker or "Голос", fragment, start, end)


class Library:
    def __init__(self, directory: Path = DIRECTORY):
        catalog = json.loads((directory / "catalog.json").read_text(encoding="utf-8"))
        self.revision = catalog["revision"]
        self.repository = catalog["repository"]
        self.books = {b.key: b for b in (Book(**entry) for entry in catalog["works"])}
        self.entries: list[Passage] = []
        for book in self.books.values():
            raw = (directory / book.file).read_bytes()
            if hashlib.sha256(raw).hexdigest() != book.sha256:
                raise ValueError(f"Контрольная сумма не совпала: {book.file}")
            self.entries.extend(passages(book, raw))
        self.positions = {p.ref: n for n, p in enumerate(self.entries)}
        self.postings = defaultdict(dict)
        self.lengths = []
        self.bodies = []
        self.names = set()
        self.book_terms = defaultdict(set)
        for i, p in enumerate(self.entries):
            body = tokens(p.text)
            counts = Counter(body)
            if p.speaker != "Ремарка":
                name = tokens(p.speaker)
                counts.update(name)
                self.names.update(name)
            self.bodies.append(set(counts))
            self.book_terms[p.book].update(counts)
            self.lengths.append(sum(counts.values()))
            for token, freq in counts.items():
                self.postings[token][i] = freq
        self.avg_length = sum(self.lengths) / max(1, len(self.entries))
        self.synonyms = {}
        for group in GROUPS:
            words = set(tokens(group))
            for word in words:
                self.synonyms[word] = words - {word}

    def get(self, ref: str) -> Passage:
        if ref not in self.positions:
            raise ValueError(f"Такого фрагмента нет в каталоге: {ref}")
        return self.entries[self.positions[ref]]

    def context(self, ref: str, radius: int = 2) -> list[Passage]:
        at = self.positions[self.get(ref).ref]
        center = self.entries[at]
        if not 0 <= radius <= 5:
            raise ValueError("Радиус контекста должен быть от 0 до 5")
        return [p for p in self.entries[max(0, at-radius):at+radius+1]
                if (p.book, p.act, p.section) == (center.book, center.act, center.section)]

    def source(self, passage: Passage) -> str:
        name = self.books[passage.book].file
        return f"{self.repository}/blob/{self.revision}/tei/{name}#L{passage.first_line}-L{passage.last_line}"

    def search(self, query: str, *, book: str = "", limit: int = 3) -> list[Passage]:
        if book and book not in self.books:
            raise ValueError(f"Неизвестная книга: {book}")
        if not 1 <= limit <= 20:
            raise ValueError("Число результатов должно быть от 1 до 20")
        words = set(tokens(query))
        if book:
            meta = self.books[book]
            words -= set(tokens(meta.title + " " + meta.author))
        weights = {v: 0.62 for w in words for v in self.synonyms.get(w, ())}
        weights.update(dict.fromkeys(words, 1.0))
        vocabulary = self.book_terms[book] if book else self.postings.keys()
        if words - self.names and not (weights.keys() - self.names) & vocabulary:
            return []  # Знакомая фамилия не даёт подтверждения неизвестному предмету.
        if words - self.names:
            for name in self.names & words:
                weights[name] = 0.5
        scores = Counter()
        matched = defaultdict(set)
        info = {}
        total = len(self.entries)
        for term, weight in sorted(weights.items()):
            posting = self.postings.get(term, {})
            idf = math.log1p((total - len(posting) + 0.5) / (len(posting) + 0.5))
            info[term] = idf * weight
            for i, tf in posting.items():
                if book and self.entries[i].book != book:
                    continue
                norm = 1.2 * (0.25 + 0.75 * self.lengths[i] / self.avg_length)
                scores[i] += weight * idf * tf * 2.2 / (tf + norm)
                if term in words:
                    matched[i].add(term)
        # Два редких слова в соседних репликах могут описывать один обмен.
        # Ограничиваем прибавку и никогда не перескакиваем в другое явление.
        for i in scores:
            nearby = set()
            a = self.entries[i]
            for j in range(max(0, i-3), min(total, i+4)):
                b = self.entries[j]
                if (a.book, a.act, a.section) == (b.book, b.act, b.section):
                    nearby.update(self.bodies[j] & weights.keys())
            scores[i] += sum(weights[v] for v in sorted(matched[i]))
            scores[i] += 0.5 * sum(info[v] for v in sorted(nearby - self.bodies[i]))
        ranked = sorted(scores, key=lambda i: (-scores[i], i))
        return [self.entries[i] for i in ranked[:limit]]

    def card(self, p: Passage, query: str = "") -> dict:
        book = self.books[p.book]
        words = p.text.split()
        # Цитата в карточке — непрерывная подстрока. Для длинной реплики берём
        # короткое окно вокруг слов запроса, сохраняя авторское написание.
        wanted = set(tokens(query))
        hit = next((n for n, w in enumerate(words) if wanted.intersection(tokens(w))), 0)
        left = min(max(0, hit-5), max(0, len(words)-20))
        quote = " ".join(words[left:left+20])
        if len(words) > 20:
            sentences = [s for s in re.split(r"(?<=[.!?…])\s+", p.text)
                         if 2 <= len(s.split()) <= 20 and s.rstrip('»")').endswith((".", "!", "?", "…"))]
            if sentences:
                best = max(sentences, key=lambda s: (len(wanted.intersection(tokens(s))), len(s.split())))
                if not wanted or wanted.intersection(tokens(best)):
                    quote = best
        return {"ref": p.ref, "book": p.book, "title": book.title, "author": book.author,
                "act": p.act, "section": p.heading, "speaker": p.speaker,
                "quote": quote, "cut_before": not p.text.startswith(quote), "cut_after": not p.text.endswith(quote),
                "lines": [p.first_line, p.last_line],
                "url": self.source(p)}


In [9]:
%%theatre_module book_scene
"""Два хода с книгой: актёр выбирает свидетельство, его партнёр проверяет соседний текст."""
from collections.abc import Callable
import re

from .language import parses
from .library import Library, tokens
from .llm import Generator
from .scenario import Actor
from .schema import Delivery, Form, Lookup, Review


class Choice(Form):
    ref: str
    stance: str


def choose(gen: Generator, model: str, actor: Actor, prompt: str, cards: list[dict]) -> Choice:
    extracts = "\n".join(f"{c['ref']} · {c['speaker']}: «{c['quote']}»" for c in cards)
    return gen.ask(model, f"Ты — {actor.name}. Читаешь пьесу и готовишь короткий ответ партнёру.",
        prompt + "\nДоступные строки:\n" + extracts +
        "\nref — ровно один подходящий адрес. Если все строки посторонние, оставь пусто. "
        "stance — что сделаешь со свидетельством собеседника.",
        Choice, choices={"ref": [c["ref"] for c in cards] + [""],
                         "stance": ["уточнить", "возразить", "поддержать", "указать предел аналогии"]}, budget=160)


def find_citation(lib: Library, gen: Generator, model: str, actor: Actor, task: dict, dialogue: str,
                  event: dict, emit: Callable[[dict], None]) -> dict | None:
    book = lib.books[task["book"]]
    prompt = (f"На репетиции книги доступны целиком. Найди у {book.author} в «{book.title}» эпизод. "
              f"Задача для спора: {task['question']}\n{dialogue}\n"
              "В query дай 2–6 поисковых слов: участники книжного эпизода, предмет или действие. "
              "Не используй имена актёров нынешней постановки.")
    for attempt in (1, 2):
        query = gen.ask(model, f"Ты — {actor.name}, работаешь с театральной библиотекой.",
                        prompt, Lookup, budget=120).query.strip()
        hits = [lib.card(p, query) for p in lib.search(query, book=book.key)]
        picked = choose(gen, model, actor, task["question"], hits) if hits else None
        ref = picked.ref if picked else ""
        emit({**event, "event": "book_search", "tool": "search_library", "attempt": attempt,
              "book": book.key, "query": query, "hits": hits, "selected": ref})
        if ref:
            card = next(c for c in hits if c["ref"] == ref)
            return {**event, "event": "citation", "kind": "read", "card": card,
                    "stance": picked.stance, "question": task["question"], "motif": task["motif"]}
        prompt += f"\nПо запросу «{query}» подходящий фрагмент не выбран. Переформулируй другими словами."
    return None


def check_citation(lib: Library, gen: Generator, model: str, actor: Actor, published: dict,
                   event: dict, emit: Callable[[dict], None]) -> dict | None:
    first = published["card"]
    context = lib.context(first["ref"], radius=3)
    cards = [lib.card(p) for p in context]
    others = [c for c in cards if c["ref"] != first["ref"]]
    prompt = (f"Партнёр прочитал у {first['author']} «{first['quote']}». "
              f"Обсуждали: {published['question']} Сейчас проверь чужое чтение. "
              "Выбери соседнюю строку, которая уточняет, меняет или ограничивает вывод.")
    picked = choose(gen, model, actor, prompt, others) if others else None
    ref = picked.ref if picked else ""
    emit({**event, "event": "book_context", "tool": "open_context", "origin": first["ref"],
          "origin_turn": published["index"], "hits": cards, "selected": ref})
    if not ref:
        return None
    return {**event, "event": "citation", "kind": "check", "card": next(c for c in others if c["ref"] == ref),
            "origin": first["ref"], "origin_turn": published["index"], "stance": picked.stance,
            "question": published["question"], "motif": published["motif"]}


def repeats_partner(line: str, previous: str) -> bool:
    # Регистром или потерянной при монтаже кавычкой можно замаскировать целый
    # чужой вопрос. Короткое «Да, я согласен» остаётся допустимым ответом.
    def tokens_of_phrase(text: str) -> list[str]:
        return re.findall(r"\w+", text.casefold().replace("ё", "е"))
    heard = tokens_of_phrase(previous)
    return len(heard) >= 5 and tokens_of_phrase(line) == heard


def check_comment(words: Delivery, citation: dict, cast_text: str, previous: str = "") -> Review:
    card = citation["card"]
    if repeats_partner(words.line, previous):
        return Review(issue="book", note="Ты повторил весь вопрос партнёра. Ответь на него своими словами.")
    if (words.line.count("«") != words.line.count("»")
            or re.search(r"(?:с уч[её]том(?: е[её]| его| их)?|потому что|чтобы)\s*(?:\.{3}|…)\s*$", words.line, re.I)):
        return Review(issue="book", note="Фраза оборвалась. Договори мысль одним целым предложением.")
    if re.search(r"[a-z]{2,}", words.line, re.I | re.ASCII):
        return Review(issue="book", note="Весь комментарий должен прозвучать по-русски.")
    if len(words.line.split()) > 36:
        return Review(issue="book", note="Оставь одно-два коротких предложения на русском, всего до 30 слов.")
    def word_forms(text):
        forms = set(tokens(text))
        # У фамилии «Хлестаков» первый разбор без контекста бывает «хлестак»,
        # тогда как у «Хлестакову» первым идёт разбор фамилии.
        for word in re.findall(r"\b[А-ЯЁ][а-яё]+\b", text):
            forms.add(word.casefold().replace("ё", "е"))
            forms.update(p.normal_form.replace("ё", "е") for p in parses(word.casefold())
                         if {"Name", "Surn", "Patr"} & p.tag.grammemes)
        return forms
    support = word_forms(citation["motif"] + " " + card["speaker"] + " " + card["quote"])
    known = support | word_forms(cast_text + " " + citation["question"] + " " + card["title"] + " " + card["author"])
    for word in re.findall(r"\b[А-ЯЁ][а-яё]+\b", words.line):
        parsed = parses(word.casefold())[0]
        surname = re.search(r"(?:ов|ев|ин|ова|ева|ский|ская|ского|скому|овым|евым|ович|овна)$", word, re.I)
        if (({"Name", "Surn", "Patr"} & parsed.tag.grammemes or surname and parsed.tag.POS in {"NOUN", "ADJF"})
                and word.casefold().replace("ё", "е") not in known
                and not any(p.normal_form.replace("ё", "е") in known for p in parses(word.casefold()))):
            return Review(issue="book", note=f"Имя «{word}» не подтверждено открытой карточкой или составом сцены.")
    for quote in re.findall(r'«([^»]+)»|"([^"]+)"', words.line):
        text = next(q for q in quote if q)
        title = len(text.split()) == len(card["title"].split()) and tokens(text) == tokens(card["title"])
        if len(text.split()) > 1 and not title and text.casefold() not in card["quote"].casefold():
            return Review(issue="book", note=f"Убери неподтверждённую цитату «{text}». Выскажи свой довод своими словами, без кавычек.")
    if not support.intersection(word_forms(words.line)):
        return Review(issue="book", note=f"Свяжи свой ответ с конкретной деталью из книги: {citation['motif']}.")
    addressed = re.search(r"\b(я|ты|вы|мы|нам|мне|меня|нас|тебе|тебя|тобой|вам|вас|вами|здесь|наш\w*|ваш\w*|"
                          r"давайте|предлагаю|хочу|отказываюсь|надо|нужно|спросим|нельзя|пусть)\b", words.line, re.I)
    imperative = any("impr" in parses(w.casefold())[0].tag for w in re.findall(r"[А-Яа-яЁё]+", words.line))
    if "?" not in words.line and not addressed and not imperative:
        return Review(issue="book", note="Обратись к своим партнёрам: что именно нам здесь делать после прочитанного?")
    return Review(issue="", note="")


def shorten_comment(words: Delivery, citation: dict, previous: str = "") -> Delivery:
    if len(words.line.split()) <= 36:
        return words
    # Сохраняем полные предложения исходного актёрского дубля. Превращение
    # длинной фразы в оборванную цитату при монтаже слышно особенно отчётливо.
    sentences = [s.strip() for s in re.findall(r'[^.!?…]+(?:[.!?…]+[»"]*)', words.line)]
    candidates = [s for s in sentences if 5 <= len(s.split()) <= 30
                  and not s.rstrip('»"').endswith(("...", "…"))
                  and s.count("«") == s.count("»") and not repeats_partner(s, previous)]
    if not candidates:
        return words
    motif = set(tokens(citation["motif"]))
    best = max(candidates, key=lambda s: (bool(motif.intersection(tokens(s))), "?" in s,
                                          bool(re.search(r"\b(мы|нам|я|ты|вы|наш)\b", s, re.I))))
    return words.model_copy(update={"line": best})


def acting_note(citation: dict) -> str:
    card = citation["card"]
    return (f"Перед твоей репликой зрители услышат подлинный отрывок из «{card['title']}», "
            f"{card['speaker']}: «{card['quote']}». Задача: {citation['question']} "
            f"Твой способ ответа: {citation['stance']}. Одной живой фразой свяжи конкретную деталь из книги "
            "с выбором присутствующего. Если не согласен с аналогией, объясни почему. В книжном эпизоде "
            "вы не участвовали. Цитату прочитает постановка, свой комментарий скажи без кавычек.")


def citation_intro(event: dict) -> str:
    card = event["card"]
    action = "проверяет соседний фрагмент" if event["kind"] == "check" else "открывает книгу"
    speaker = "Авторская ремарка" if card["speaker"] == "Ремарка" else f"В книге говорит {card['speaker']}"
    return f"{event['name']} {action}. «{card['title']}», действие {card['act']}. {speaker}."


def printed_quote(card: dict) -> str:
    quote = card["quote"]
    before = "…" if card.get("cut_before") and quote[:1].islower() else ""
    after = "…" if card.get("cut_after") and not quote.rstrip('»")').endswith((".", "?", "!", "…")) else ""
    return before + quote + after


In [10]:
%%theatre_module stage_tools
"""Сценические инструменты. Актёр выбирает вызов до того, как решит, что сказать."""

from collections.abc import Callable
from dataclasses import asdict

from .archive import Archive, Hit, terms
from .llm import Generator
from .scenario import Scenario
from .schema import ToolCall, Turn


def quorum(option: str, cast_size: int) -> int:
    if option == "challenge":
        return cast_size
    if option == "nomination":
        return cast_size // 2 + 1
    raise ValueError(f"Неизвестный вид голосования: {option}")


def recall_line(history: list[Turn], speaker: str, query: str) -> Turn | None:
    """Ищем среди принятых реплик этого героя за все действия. При ничьей берём последнюю."""
    matches = [t for t in reversed(history) if t.actor == speaker]
    words = terms(query) - {"после", "репли", "фраза", "фразу", "слова", "сказа"}
    if not matches or not words:
        return next(iter(matches), None)
    scored = [(len(words & terms(t.speech.line)), t) for t in matches]
    score, turn = max(scored, key=lambda item: item[0])
    return turn if score else None


def use_skill(gen: Generator, model: str, story: Scenario, actor: str, scene: int, context: str,
              history: list[Turn], archive: Archive, searched: set[str], seen: dict[str, Hit],
              emit: Callable[[dict], None]) -> str:
    speakers = list(dict.fromkeys(t.actor for t in history))
    actions = ["", *(f"recall_line:{key}" for key in speakers)]
    menu = []
    if scene in (2, 3) and actor in story.researchers and actor not in searched:
        actions.append("search_archive")
        menu.append("search_archive — уточнить условие по файлу. Опись: " + " / ".join(archive.titles))
    if speakers:
        menu.append("recall_line:код — уточнить, что этот человек действительно сказал. "
                    "В query укажи тему или оставь пустым для его последней реплики. Коды: "
                    + ", ".join(f"{key} = {story.cast[key].name}" for key in speakers))
    if scene >= 2:
        actions.extend(("test_ballot:challenge", "test_ballot:nomination"))
        menu.append("test_ballot:challenge — проверить условную поддержку совместного пересмотра, "
                    "test_ballot:nomination — поддержку одного имени. В supporters поставь число от 1 до 4.")
    if not menu:
        return ""
    call = gen.ask(model, f"Ты — {story.cast[actor].name}. Выбираешь сценический инструмент перед ходом.",
        context + "\n" + "\n".join(menu) +
        "\nКакую одну деталь стоит проверить для ответа партнёру? Выбери action. "
        "Для поиска в архиве нужны ключевые слова из описи. Для проверки голосов достаточно числа. "
        "Если проверять нечего, оставь action пустым.", ToolCall, choices={"action": actions}, budget=175)
    if not call.action:
        return ""
    index = len(history) + 1
    event = {"scene": scene, "index": index, "actor": actor, "model": model}
    if call.action == "search_archive":
        searched.add(actor)
        hits = archive.search(call.query)
        seen.update((hit.ref, hit) for hit in hits)
        emit({**event, "event": "search", "tool": call.action, "query": call.query,
              "hits": [asdict(h) for h in hits]})
        return "В найденном документе: " + " ".join(h.text for h in hits) if hits else "Архив не нашёл подтверждения."
    tool, argument = call.action.split(":", 1)
    if tool == "recall_line":
        turn = recall_line(history, argument, call.query)
        args = {"speaker": argument, "query": call.query}
        result = {"turn": turn.index, "scene": turn.scene, "text": turn.speech.line} if turn else None
        message = (f"Точные слова {story.cast[argument].name} из действия {turn.scene}: «{turn.speech.line}»."
                   if turn else "Подходящей реплики этого участника в записи нет.")
    else:
        needed = quorum(argument, len(story.cast))
        args = {"option": argument, "supporters": call.supporters}
        result = {"required": needed, "enough": call.supporters >= needed}
        option = "совместный пересмотр" if argument == "challenge" else "выбор одного имени"
        message = (f"Мысленный пример: за {option} — {call.supporters} из {len(story.cast)}. "
                   f"Порог — {needed}. Голосов {'достаточно' if result['enough'] else 'пока мало'}. "
                   "Это условный расчёт. Фактические голоса регистрируются отдельно.")
    emit({**event, "event": "skill", "tool": tool, "arguments": args, "result": result})
    return message


## Макроплан, очередь актёров и репетиция

Драматург получает конфликт и четыре этапа до первого выхода актёра. На каждом ходу режиссёр учитывает длительное молчание, инициативу новой сцены и адресата последней реплики. Актёр выбирает инструмент и намерение, играет дубль, при необходимости получает замечание. В финале у каждого ровно один бюллетень.

In [11]:
%%theatre_module rehearsal
"""Проверки черновика до выхода на сцену. Замечания адресуются той же модели."""

import re
from collections.abc import Callable, Sequence
from difflib import SequenceMatcher

from .archive import terms
from .llm import GenerationError, Generator
from .language import wrong_gender
from .scenario import Actor
from .schema import Delivery, EditorialCheck, Review


class BookCommentRejected(GenerationError):
    """Актёр не сумел подтвердить свой комментарий и может отказаться от публичной цитаты."""


FAMILY_FORMS = {
    "брат": r"брат(?:а|у|ом|е|ья|ьев)?", "сестра": r"сестр(?:а|у|ы|е|ой|ёнка)",
    "мать": r"(?:мать|матери|мам(?:а|у|ы|е|ой))", "отец": r"(?:отец|отц[ауы]|пап[ауы])",
    "родители": r"родител(?:и|ей|ям|ях)", "дети": r"(?:дети|детей|детям|ребёнок|ребёнка)",
    "жена": r"жен[ауые]|супруга", "муж": r"муж(?:а|у|ем)?",
    "сын": r"сын(?:а|у|ом|овья)?", "дочь": r"(?:дочь|доч(?:ка|ери|ку))",
}
FAMILY_CLAIM = re.compile(
    r"\b(?:(?:мо[йяиюё]|моего|моей|моих)\s+|у меня (?:есть |был[аи]? )?)"
    r"(?:(?:старш|младш|родн)\w+\s+)?(?:"
    + "|".join(f"(?P<{relation}>{forms})" for relation, forms in FAMILY_FORMS.items())
    + r")\b", re.I,
)
INLINE_ACTION = re.compile(
    r"[(（*]\s*(?:я\s+)?(?:показ|подним|подня|покач|поворач|наклон|смотр|гляд|"
    r"сжим|указ|вздых|пожим|улыба|прикас|отступ|подход|шепч|кива)\w*[\s,.)*]", re.I,
)
SELF_VOTE_CONFLICT = re.compile(
    r"\b(?:не могу (?:их|семью|детей|Монику) (?:оставить|бросить)|"
    r"(?:долж(?:ен|на)|обязан[а]?) (?:сам[а]? )?вернуться домой)\b", re.I,
)

# Формулировки адресованы актёру. Причины и исходные реплики также попадут в журнал.
NOTES = {
    "family": "Убери личную историю о родственнике, которого нет в твоей карточке. Возьми мотив из роли.",
    "vote": "Ты согласился остаться здесь. Объясни, ради чего принимаешь разлуку с домом.",
    "repeat": "Ты уже высказывал этот довод почти теми же словами. Ответь партнёру поступком, уступкой или контрпримером.",
    "clock": "Часы уже обсуждали двое. Оставь этот предмет за кадром. Что объявленное условие делает с конкретным человеком?",
    "aside": "Перенеси описание движения из прямой речи в поле жеста. Вслух оставь только слова.",
    "length": "Сократи реплику до двух коротких предложений, жест — до одного движения.",
    "gender": "Согласуй род слов о себе и адресате с карточками ролей.",
    "unfinished": "Мысль оборвалась. Произнеси законченное короткое предложение с точкой или вопросительным знаком.",
}


def mentions_clock(line: str) -> bool:
    return bool(re.search(r"\bчас(?:ы|ов|ам|ами|ах)\b", line, re.I))


def draft_issues(words: Delivery, actor: Actor, prior: list[str], staying: bool,
                 clock_discussed: bool = False, partners: Sequence[Actor] = (),
                 addressee: Actor | None = None) -> list[str]:
    """Узкие признаки для переигровки, а не оценка литературного качества."""
    issues = []
    if wrong_gender(words.line, actor, partners, addressee):
        issues.append("gender")
    ending = words.line.rstrip('»"”’*) ')
    dangling = r"\b(?:если|чтобы|потому что|но|и|что я|если (?:вы собираетесь|ты собираешься))[.!?…]+$"
    if not ending.endswith((".", "?", "!", "…")) or re.search(dangling, ending, re.I):
        issues.append("unfinished")
    # Упоминание чужих родителей или цитирование партнёра само по себе допустимо.
    unquoted = re.sub(r'«[^»]*»|"[^"]*"', " ", words.line)
    for claim in FAMILY_CLAIM.finditer(unquoted):
        if claim.lastgroup not in actor.family:
            issues.append("family")
            break
    if staying and SELF_VOTE_CONFLICT.search(unquoted):
        issues.append("vote")
    vocab = terms(words.line)
    tokens = re.findall(r"\w+", words.line.casefold())
    for previous in prior[-3:]:
        earlier = terms(previous)
        shared = len(vocab & earlier)
        overlap = SequenceMatcher(None, tokens, re.findall(r"\w+", previous.casefold())).find_longest_match()
        if overlap.size >= 7 or (shared >= 4 and shared / len(vocab | earlier) >= 0.65):
            issues.append("repeat")
            break
    if clock_discussed and mentions_clock(words.line):
        issues.append("clock")
    if INLINE_ACTION.search(words.line):
        issues.append("aside")
    if len(words.line.split()) > 38 or len(words.stage.split()) > 12 or "?" in words.stage:
        issues.append("length")
    return issues


def deliver(gen: Generator, model: str, role: str, prompt: str, actor: Actor,
            prior: list[str], staying: bool, clock_discussed: bool, record: Callable[[dict], None],
            *, review: Callable[[Delivery], Review] | None = None,
            repair_prompt: str | None = None, partners: Sequence[Actor] = (),
            addressee: Actor | None = None, addressed_line: str = "",
            shorten: Callable[[Delivery], Delivery] | None = None) -> tuple[Delivery, bool, list[str]]:
    correction = ""
    for take in range(1, 4):
        words = gen.ask(model, role, prompt + correction, Delivery, budget=340)
        # Некоторые модели дословно печатают всю входную реплику перед своим ответом.
        if len(addressed_line.split()) >= 4 and words.line.startswith(addressed_line):
            own = words.line[len(addressed_line):].lstrip(" \n—")
            if own:
                words = words.model_copy(update={"line": own})
        if len(words.stage.split()) > 12:
            words = words.model_copy(update={"stage": ""})
        if shorten:
            brief = shorten(words)
            if words.line != brief.line:
                record({"take": take, "line": words.line, "stage": words.stage, "shortened_to": brief.line,
                        "issues": ["book_length"], "note": "Для чтения выбрано полное предложение из дубля."})
                words = brief
        issues = draft_issues(words, actor, prior, staying, clock_discussed, partners, addressee)
        note = ""
        if review and (not issues or take > 1):
            verdict = review(words)
            if verdict.issue:
                issues.append(verdict.issue)
                note = verdict.note
        hard = bool({"family", "vote", "biography", "ballot", "evidence", "gender", "unfinished", "book"} & set(issues))
        disputed = bool({"logic", "reply"} & set(issues))
        if not issues or (take >= 2 and not hard and (not disputed or take == 3)):
            return words, take > 1, issues
        record({"take": take, "line": words.line, "stage": words.stage, "issues": issues, "note": note})
        if take == 3:
            if "book" in issues:
                raise BookCommentRejected(note)
            raise GenerationError(f"{actor.name}: после трёх дублей остаётся замечание {', '.join(issues)}")
        if hard and repair_prompt:
            # Повтор забракованного анекдота в подсказке закреплял его у малой модели.
            # При фактической ошибке актёр сохраняет адресата, источник и голос,
            # но получает чистый лист вместо черновика и своего прежнего намерения.
            prompt = repair_prompt
            hints = {
                "biography": "Используй только дословно подтверждаемую личную деталь из карточки роли. Можно прямо отказаться предъявлять воспоминание.",
                "evidence": "Скажи лишь то, что действительно установлено. Следующий шаг сформулируй как предложение или вопрос.",
                "ballot": "Выскажи одну причину уже принятого решения и назови его цену лично для тебя.",
                "reply": "Ответь на конкретный вопрос партнёра одной фразой. Можно признать незнание или несогласие.",
                "logic": "Исправь противоречие с уже сказанным. Объясни, что в словах партнёра меняет твой выбор.",
            }
            correction = "\n" + "\n".join(hints.get(issue, NOTES.get(issue, "")) for issue in issues)
            if note and any(issue in {"reply", "logic", "ballot", "book"} for issue in issues):
                correction += "\nУточнение редактора: " + note
            if take == 2:
                correction += "\nОставь только одно простое предложение, которое можно обосновать карточкой."
            continue
        instructions = [NOTES[issue] for issue in issues if issue in NOTES]
        if note:
            instructions.append(note)
        correction = (f"\nРедактор возвращает твой черновик: «{words.line}»\n"
                      + "\n".join(instructions) + "\nСыграй заново одной короткой фразой.")
    raise AssertionError("unreachable")


def review_line(gen: Generator, editor: str, actor: Actor, context: str, cue: str,
                words: Delivery, *, addressed_line: str, final: bool) -> Review:
    categories = ["ok", "biography", "evidence"]
    if addressed_line:
        categories.extend(["logic", "echo"])
        if "?" in addressed_line:
            categories.append("reply")
    if final:
        categories.append("ballot")
    verdict = gen.ask(editor,
        "Ты принимаешь актёрские дубли. По умолчанию ставь ok. Менять дубль стоит только при "
        "конкретной ошибке. Персонажи вправе спорить, строить гипотезы, шутить и задавать встречные вопросы.",
        f"Роль: {actor.name}. Биография: {actor.stake}\nСцена и известные записи:\n{context}\n"
        f"Выбор актёра: {cue}\nСлова, на которые отвечаем: {addressed_line or '(сцену только открыли)'}\n"
        f"Оцени только эту прямую речь: «{words.line}».\n"
        "ok — принять. biography — новое неподтверждённое конкретное событие личного прошлого. "
        "evidence — объявлено фактом измерение, которого в сцене не было. logic — явное "
        "противоречие с конкретным утверждением или приписывание партнёру чужих слов. "
        "reply — прямой вопрос проигнорирован. Признание незнания и уточнение по теме допустимы. "
        "echo — дословный повтор без развития. ballot — призыв, противоположный принятому голосу. "
        "При ok оставь excerpt, anchor и note пустыми. При ошибке в excerpt дословно скопируй "
        "проблемные слова актёра, в anchor — точную цитату вопроса, условия или голоса. "
        "В note коротко укажи, что исправить. Не оценивай жест и литературный вкус.",
        EditorialCheck, choices={"issue": categories}, budget=390)
    def normalize(value: str) -> str:
        return " ".join(value.casefold().replace("ё", "е").strip(' «»"').split())
    excerpt = normalize(verdict.excerpt)
    reference = normalize(verdict.anchor)
    source = addressed_line if verdict.issue == "reply" else context + "\n" + cue
    if (verdict.issue == "ok" or not excerpt or excerpt not in normalize(words.line)
            or (verdict.issue in {"reply", "logic", "ballot"}
                and (not reference or reference not in normalize(source)))):
        return Review(issue="", note="")
    return Review(issue=verdict.issue, note=verdict.note)


In [12]:
%%theatre_module engine
"""Один вызов актёра — один ход. Актёры узнают чужие позиции из произнесённых реплик."""

from __future__ import annotations

from collections import Counter
from collections.abc import Callable
from functools import partial
import re

from .archive import Archive, Hit
from .book_scene import acting_note, check_citation, check_comment, find_citation, shorten_comment
from .library import Library
from .language import stage_direction
from .llm import Generator
from .rehearsal import BookCommentRejected, deliver, mentions_clock, review_line
from .scenario import Scenario
from .schema import Intention, Plan, Speech, Turn
from .stage_tools import quorum, use_skill


def choose_speaker(cast: list[str], history: list[Turn], eligible: list[str],
                   opener: str | None = None, *, new_scene: bool = False) -> tuple[str, str]:
    """Прямой вопрос важнее очереди, пока он не вытесняет остальных из сцены."""
    if not eligible:
        raise ValueError("нет доступных актёров")
    last = history[-1] if history else None
    candidates = [a for a in eligible if last is None or a != last.actor] or eligible
    last_seen = dict.fromkeys(cast, -1)
    for index, turn in enumerate(history):
        last_seen[turn.actor] = index
    oldest = min(candidates, key=last_seen.__getitem__)
    if len(history) - 1 - last_seen[oldest] >= len(cast) + 1:
        return oldest, "давно не получал слова"
    if opener in candidates:
        return opener, "инициатива в новой сцене"
    if not new_scene and last and last.speech.to in candidates:
        return last.speech.to, f"обращение от {last.actor}"
    return oldest, "дольше других ждёт свободного слова"


def resolve(story: Scenario, votes: dict[str, str]) -> tuple[str, str]:
    if set(votes) != set(story.cast):
        raise ValueError("нужны решения всех четырёх персонажей")
    allowed = {"challenge", "abstain", *(f"select:{key}" for key in story.cast)}
    if not set(votes.values()) <= allowed:
        raise ValueError("неизвестный вариант решения")
    count = Counter(votes.values())
    if count["challenge"] >= quorum("challenge", len(votes)):
        return "challenge", story.unanimous
    choice, n = count.most_common(1)[0]
    if choice.startswith("select:") and n >= quorum("nomination", len(votes)):
        name = story.cast[choice.removeprefix("select:")].name
        return choice, story.nomination.format(name=name)
    return "unresolved", story.unresolved


def make_plan(story: Scenario, gen: Generator, model: str) -> Plan:
    cast = ", ".join(actor.name for actor in story.cast.values())
    return gen.ask(model, "Ты драматург русскоязычной камерной пьесы.",
        f"Пьеса «{story.title}». {story.setting}\nВ ролях: {cast}.\n"
        f"Проблема: {story.question}\nОбъявление во второй сцене: {story.notice}\n"
        "Составь макроплан до игры: ровно четыре сцены в этом порядке.\n"
        + "\n".join(f"{i}. {beat} Действие на сцене: {action}"
                    for i, (beat, action) in enumerate(zip(story.arc, story.blocking, strict=True), 1))
        + "\nДля каждой придумай title (заглавие), opening (одно короткое движение, свет или звук "
          "до слов героев, до 15 слов), objective (конкретная задача этого спора). Сохрани реквизит. "
          "Текст табло выведет сама программа. Оставь за кадром надписи, готовые реплики, "
          "новых героев и предсказания голосования. Во всех четырёх opening дверь ещё закрыта. "
          "Напиши по-русски объект со списком scenes.", Plan, budget=900)


def history_text(story: Scenario, history: list[Turn]) -> str:
    lines = []
    for t in history[-5:]:
        gesture = f" ({t.speech.stage})" if t.speech.stage else ""
        lines.append(f"{story.cast[t.actor].name}{gesture}: {t.speech.line}")
    return "\n".join(lines)


def pending_question(actor: str, history: list[Turn]) -> Turn | None:
    """Вопрос остаётся адресованным герою, даже если режиссёр дал слово третьему."""
    for turn in reversed(history):
        if turn.actor == actor:
            break
        if turn.speech.to == actor and "?" in turn.speech.line:
            return turn
    return None


def named_addressee(line: str, story: Scenario, candidates: list[str]) -> str | None:
    """Партнёр, которого позвали вслух, может отличаться от первоначального намерения."""
    for key in candidates:
        names = [story.cast[key].name, *story.cast[key].name.split()]
        forms = "|".join(re.escape(n) for n in names)
        if re.search(rf"(?:^|[.!?,]\s+)(?:мистер |мисс )?(?:{forms})\s*[,!?.]", line, re.I):
            return key
    return None


def ballot_phrase(story: Scenario, actor: str, vote: str) -> str:
    """Стандартная формула регистрации голоса. Обоснование пишет сам актёр."""
    if vote == "abstain":
        return "Я воздержусь."
    if vote == "challenge":
        return ("Я за совместный пересмотр." if story.key == "extra"
                else "Я отказываюсь определять, кто из нас настоящий.")
    nominee = vote.removeprefix("select:")
    if nominee == actor:
        return "Я остаюсь здесь." if story.key == "extra" else "Я называю себя."
    return f"Мой выбор — {story.cast[nominee].name}."


def play_turn(gen: Generator, model: str, story: Scenario, actor_id: str, context: str,
              seen: dict[str, Hit], eligible: list[str], final: bool, prior: list[str],
              clock_discussed: bool, record: Callable[[dict], None], editor: str | None,
              anchor: Turn | None = None, citation: dict | None = None) -> Speech:
    """Актёр сначала принимает решение, затем говорит, не видя служебных полей и путей."""
    actor = story.cast[actor_id]
    role = (f"Ты — {actor.name}. Играешь в русскоязычной камерной трагикомедии. "
            f"О себе говори в форме «я» ({actor.agreement}). {actor.voice}\nТвоя личная история в этой постановке: {actor.stake}\n"
            f"Твой способ вести спор: {actor.tactic}\n"
            f"Пример ритма речи: «{actor.sample}». "
            "Личное прошлое бери только из своей истории. Можно сомневаться, уступать и менять мнение.")
    labels = ", ".join(f"{key} = {a.name} ({a.agreement})" for key, a in story.cast.items())
    notes = "\n".join(f"{ref} — {h.text}" for ref, h in seen.items())
    decisions = {"challenge": story.challenge, "abstain": "воздержаться от решения"}
    for key, a in story.cast.items():
        meaning = "оставить в комнате" if story.key == "extra" else "признать настоящим"
        decisions[f"select:{key}"] = f"{meaning} одного участника ({a.name})"
    vote_rules = ""
    if final:
        vote_rules = ("Вспомни обещание, которое ты дал вслух, и предложение партнёра. Голосуй в "
                      "соответствии со своей позицией. Если передумал, в реплике потребуется причина. "
                      "Четыре возражения открывают пересмотр; "
                      "три одинаковых имени принимаются терминалом. Значения поля vote:\n"
                      + "\n".join(f"{k} — {v}" for k, v in decisions.items()))
    tactics = ["ответить прямо", "возразить", "предложить проверку", "уступить", "признать незнание",
               "уточнить сказанное", "объяснить личную цену"]
    move = gen.ask(model, role,
        context + "\nПодготовь свой ход за кулисами. В intent выбери способ ответа партнёру. "
        f"to — адресат или пусто. Участники: {labels}.\n"
        f"Твои архивные карточки:\n{notes or '(пока пусто)'}\n"
        f"source — одна карточка, на которую опираешься, иначе пусто. {vote_rules}",
        Intention, choices={"intent": tactics, "to": ["", *(key for key in eligible if key != actor_id)],
                            "source": ["", *seen], "vote": list(decisions) if final else [""]},
        budget=330)
    cue = f"Способ вести эту реплику — {move.intent}."
    if move.to:
        partner = story.cast[move.to]
        cue += f" После ответа можно передать слово: {partner.name} ({partner.agreement})."
    binding = f"Следующий адресат — {story.cast[move.to].name}." if move.to else "Обратись к присутствующим."
    if final:
        # В намерении могут фигурировать чужие предложения. Для финального обоснования
        # передаём только зарегистрированный выбор, особенно если актёр выбрал себя.
        pledge = ballot_phrase(story, actor_id, move.vote)
        cue = (f"Твой голос уже принят: «{pledge}». Выбранное действие: {decisions[move.vote]}. "
               "Произнеси только причину: что ты готов потерять или за что берёшь ответственность? "
               "Формулу выбора перед причиной добавит табло.")
        if story.key == "extra" and move.vote == f"select:{actor_id}":
            cue += " Если решение примут, остальные уйдут, а ты останешься в комнате. Осознай эту цену."
        binding = cue
    if move.source:
        fact = f" На твоей карточке: «{seen[move.source].text}». Используй один точный факт из неё в обосновании."
        cue += fact
        binding += fact
    prompt = (context + f"\n{cue}\nСыграй живой разговор. Сначала коротко ответь на то, что партнёр спросил "
              "или предложил. Затем, если нужно, добавь одно действие, сомнение или уступку. "
              "Когда ответа нет, признай это. Не приписывай партнёру слов, которых он не произносил. "
              "Одна законченная реплика до 30 слов. Имя собеседника можно опустить. "
              "Вслух говори только за себя. По желанию добавь отдельно одно видимое движение "
              "от третьего лица. За экран и дверь отвечает постановщик.")
    review = partial(review_line, gen, editor, actor, context, binding,
                     addressed_line=anchor.speech.line if anchor else "", final=final) if editor else None
    clean = (context + f"\n{binding}\nСыграй новый короткий ответ. Факты твоей жизни: {actor.stake} "
             "Если этих фактов недостаточно, скажи партнёру, почему доказательства пока нет. "
             "Вслух говори только за себя. Жест можно опустить.")
    if citation:
        cast_text = " ".join(a.name + " " + a.stake for a in story.cast.values())
        previous_line = anchor.speech.line if anchor else ""
        ordinary_review = review
        def book_review(words):
            verdict = check_comment(words, citation, cast_text, previous_line)
            return ordinary_review(words) if not verdict.issue and ordinary_review else verdict
        review = book_review
        role = f"Ты — {actor.name}, {actor.agreement}. {actor.voice} Твоя личная история: {actor.stake}"
        last = f"Перед этим партнёр сказал: «{anchor.speech.line}»." if anchor else ""
        prompt = (f"Вы спорите вчетвером. {story.notice} {last}\n{acting_note(citation)}\n"
                  f"Детали для разговора: {citation['motif']}. "
                  "Книжный текст уже прочитан. Теперь скажи, что ТЫ предлагаешь сделать здесь, "
                  "в связи с одной услышанной деталью. Один короткий ответ или вопрос нынешнему партнёру. "
                  "Только своими словами, без кавычек, до 25 слов. Закулисные метки не произносить. "
                  "Жест можно опустить.")
        clean = prompt
    words, edited, issues = deliver(gen, model, role, prompt, actor, prior,
                                    final and story.key == "extra" and move.vote == f"select:{actor_id}",
                                    clock_discussed, record, review=review, repair_prompt=clean,
                                    partners=[a for key, a in story.cast.items() if key != actor_id],
                                    addressee=story.cast.get(move.to or (anchor.actor if anchor else "")),
                                    addressed_line=anchor.speech.line if anchor else "",
                                    shorten=partial(shorten_comment, citation=citation, previous=previous_line) if citation else None)
    line = words.line
    if final and not line.startswith(pledge):
        line = f"{pledge} {line}"
    partners = [key for key in eligible if key != actor_id]
    addressed = named_addressee(words.line, story, partners) or move.to
    gesture = stage_direction(words.stage, actor) if words.stage != words.line and len(words.stage.split()) <= 16 else ""
    return Speech(line=line, stage=gesture, to=addressed, source=move.source, vote=move.vote,
                  edited=edited, quality_issues=issues)


def perform(story: Scenario, gen: Generator, models: dict[str, str], planner: str,
            turns_per_scene: int, emit: Callable[[dict], None], *, plan: Plan | None = None,
            editor: str | None = None, library: Library | None = None) -> None:
    """Опубликованные события образуют общий журнал. Готовый план позволяет повторить опыт."""
    if not 4 <= turns_per_scene <= 10:
        raise ValueError("для первых трёх сцен допустимо 4–10 ходов на сцену")
    if set(models) != set(story.cast) or len(set(models.values())) < 2:
        raise ValueError("назначьте каждому актёру модель, всего хотя бы две разные")
    cast = list(story.cast)
    archive = Archive(story.archive)
    notebooks: dict[str, dict[str, Hit]] = {key: {} for key in cast}
    history: list[Turn] = []
    public_readings: list[dict] = []
    votes: dict[str, str] = {}
    scene_openers = {1: cast[0], 2: story.researchers[-1], 3: "chandler"}

    reused = plan is not None
    if plan is None:
        plan = make_plan(story, gen, planner)
    emit({"event": "plan", "model": planner, "reused": reused, **plan.model_dump()})
    for scene, (beat, outline) in enumerate(zip(story.arc, plan.scenes, strict=True), 1):
        final = scene == len(plan.scenes)
        action = story.blocking[scene - 1]
        emit({"event": "scene", "scene": scene, "title": outline.title,
              "opening": outline.opening, "action": action, "notice": story.notice if scene == 2 else ""})
        checked: set[str] = set()
        public_citation = None
        previous_arguments: dict[str, Turn] = {}
        if scene >= 3:
            for turn in history:
                if turn.scene == scene - 1:
                    previous_arguments[turn.actor] = turn
        for index in range(len(cast) if final else turns_per_scene):
            eligible = [key for key in cast if key not in votes]
            candidates = eligible
            if scene == 1:
                unheard = [key for key in cast if all(t.actor != key for t in history)]
                if turns_per_scene - index <= len(unheard):
                    candidates = unheard
            # Во второй сцене табло даёт повод исследователю первым проверить условие.
            opener = scene_openers.get(scene)
            actor_id, reason = choose_speaker(cast, history, candidates, opener if index == 0 else None,
                                              new_scene=index == 0)
            actor = story.cast[actor_id]
            model = models[actor_id]
            context = (f"Ты — {actor.name}. {story.setting}\nСейчас сцена {scene}: {outline.title}. "
                       f"{beat}\nНа сцене сейчас: {outline.opening}\nУже произошло при всех: {action}\n"
                       f"Задача сцены: {outline.objective}\n")
            if scene >= 2:
                context += f"Сообщение на табло: {story.notice}\n"
            in_scene = [turn for turn in history if turn.scene == scene]
            if previous_arguments:
                context += ("Что участники успели сказать в прошлом действии:\n"
                            + history_text(story, list(previous_arguments.values())) + "\n")
            # Новое объявление меняет разговор. При переходе сохраняем одну последнюю
            # реплику, дальше актёр слышит партнёров текущей сцены.
            heard = in_scene or ([] if previous_arguments else history[-1:])
            context += "Последние слова на сцене:\n" + (history_text(story, heard) or "(тишина)") + "\n"
            if public_readings:
                context += "Уже прочитали вслух при всех:\n" + "\n".join(
                    f"{e['name']} открывает «{e['card']['title']}», в книге {e['card']['speaker']}: "
                    f"«{e['card']['quote']}»." for e in public_readings[-2:]) + "\n"
            prior = [turn.speech.line for turn in history if turn.actor == actor_id]
            question = pending_question(actor_id, in_scene)
            cues = story.cues[scene - 1]
            cue = cues[min(index * len(cues) // turns_per_scene, len(cues) - 1)]
            context += (f"Направление действия: {cue}\nДверь пока закрыта. Гипотезу или предложение "
                        "формулируй как гипотезу. Действие или результат измерения можно считать "
                        "состоявшимся лишь при наличии публичной ремарки или записи.\n")
            if scene == 3 and index >= turns_per_scene - 2:
                context += ("До голосования осталось время договориться. Если конкретного предложения "
                            "ещё нет, вынеси своё. Если партнёр его уже сделал, скажи, поддерживаешь ли "
                            "ты его и на каких условиях. Обещай только за себя.\n")
            # При смене действия новое объявление имеет приоритет. Внутри действия
            # прерванный прямой вопрос важнее последней произнесённой фразы.
            anchor = question or (in_scene[-1] if in_scene else None)
            if anchor:
                context += (f"Сейчас ты отвечаешь на слова {story.cast[anchor.actor].name} "
                            f"({story.cast[anchor.actor].agreement}): «{anchor.speech.line}». "
                            "Ответь по существу, затем развивай задачу сцены. Если собеседник ошибся, "
                            "можно прямо исправить его или предложить проверку.\n")
            if final and votes:
                context += "Уже произнесённые решения:\n" + "\n".join(
                    f"{story.cast[a].name} — {ballot_phrase(story, a, vote)}" for a, vote in votes.items()) + "\n"
            seen = notebooks[actor_id]
            result = use_skill(gen, model, story, actor_id, scene, context, history, archive, checked, seen, emit)
            if result:
                context += "Что удалось проверить перед этой репликой: " + result + "\n"
            citation = None
            if library and scene in (2, 3):
                event = {"scene": scene, "index": len(history) + 1, "actor": actor_id,
                         "name": actor.name, "model": model}
                if index == 0:
                    citation = find_citation(library, gen, model, actor, story.library_tasks[scene-2],
                                             history_text(story, history[-2:]), event, emit)
                elif index == 1 and public_citation and actor_id != public_citation["actor"]:
                    citation = check_citation(library, gen, model, actor, public_citation, event, emit)
            def record(draft: dict) -> None:
                emit({"event": "book_trim" if "shortened_to" in draft else "retake", "scene": scene, "index": len(history) + 1,
                      "actor": actor_id, "model": model, **draft})

            clock_discussed = (story.key == "extra" and scene in (2, 3)
                               and len({t.actor for t in heard[-3:] if mentions_clock(t.speech.line)}) >= 2)
            play = partial(play_turn, gen, model, story, actor_id, seen=seen, eligible=eligible, final=final,
                           prior=prior, clock_discussed=clock_discussed, record=record, editor=editor, anchor=anchor)
            if citation:
                try:
                    speech = play(context + acting_note(citation), citation=citation)
                except BookCommentRejected as exc:
                    emit({**event, "event": "book_withdrawn", "ref": citation["card"]["ref"], "reason": str(exc)})
                    citation = None
                    speech = play(context)
            else:
                speech = play(context)
            if citation:
                emit(citation)
                public_citation = citation
                public_readings.append(citation)
            history.append(Turn(actor_id, scene, speech, len(history) + 1))
            if final:
                votes[actor_id] = speech.vote
            emit({"event": "turn", "scene": scene, "index": len(history), "actor": actor_id,
                  "name": actor.name, "model": model, "reason": reason, "reply_to": anchor.index if anchor else None,
                  **speech.model_dump()})
    outcome, curtain = resolve(story, votes)
    emit({"event": "ending", "outcome": outcome, "votes": votes, "curtain": curtain})


## Проверка журнала, читка и голос

Журнал связывает реплики, решения и источники. Из него строятся пьеса, сравнение условий игры и ближайшие альтернативные финалы. Литературная читка сохраняет порядок ролей и решения. Для записи Silero назначает отдельный тембр каждому актёру и рассказчице, а границы субтитров вычисляются по числу отсчётов WAV.

In [13]:
%%theatre_module analysis
"""Наблюдаемые признаки сыгранной пьесы для сравнения прогонов в ноутбуке."""

import json
from pathlib import Path

from .rehearsal import draft_issues, mentions_clock
from .scenario import load_scenario
from .schema import Delivery


def read_trace(directory: Path) -> list[dict]:
    with (directory / "trace.jsonl").open(encoding="utf-8") as source:
        return [json.loads(line) for line in source]


def summarize(events: list[dict]) -> dict[str, int]:
    story = load_scenario(events[0]["scenario"])
    turns = [event for event in events if event["event"] == "turn"]
    searches = [event for event in events if event["event"] == "search"]
    recalls = [e for e in events if e["event"] == "skill" and e["tool"] == "recall_line"]
    issues = []
    for turn in turns:
        actor = story.cast[turn["actor"]]
        staying = story.key == "extra" and turn["vote"] == "select:" + turn["actor"]
        words = Delivery(line=turn["line"], stage=turn["stage"])
        issues.extend(draft_issues(words, actor, [], staying, partners=list(story.cast.values()),
                                   addressee=story.cast.get(turn["to"])))
    return {
        "Реплик": len(turns),
        "Актёров": len({turn["actor"] for turn in turns}),
        "Успешных поисков": sum(bool(e["hits"]) for e in searches),
        "Поисков всего": len(searches),
        "Обращений к памяти диалога": len(recalls),
        "Воспоминаний с дословной цитатой": sum(e["result"] is not None for e in recalls),
        "Проверок условного голосования": sum(e.get("tool") == "test_ballot" for e in events),
        "Ходов с указанной ответной репликой": sum(t.get("reply_to") is not None for t in turns),
        "Реплик с выбранной строкой архива": sum(bool(t["source"]) for t in turns),
        "Следом говорит адресат в той же сцене": sum(
            a["scene"] == b["scene"] and a["to"] == b["actor"] for a, b in zip(turns, turns[1:])),
        "Реплик со словом «часы» в разных падежах": sum(mentions_clock(t["line"]) for t in turns),
        "Заявлений о родственнике вне карточки по шаблону": issues.count("family"),
        "Явных противоречий самовыдвижению по шаблону": issues.count("vote"),
        "Реплик с жестом внутри прямой речи по шаблону": issues.count("aside"),
        "Несогласований рода по проверяемым формам": issues.count("gender"),
        "Записанных отклонённых дублей": sum(e["event"] == "retake" for e in events),
    }


In [14]:
%%theatre_module evidence
"""Проверка книжных ссылок, небольшая разметка поиска и контрфактический разбор голосов."""
from itertools import product
import json

from .archive import terms
from .book_scene import printed_quote
from .engine import ballot_phrase, resolve
from .library import DIRECTORY, Library
from .scenario import Scenario, load_scenario


def benchmark(lib: Library) -> dict:
    cases = json.loads((DIRECTORY / "queries.json").read_text(encoding="utf-8"))
    simple = [terms(p.speaker + " " + p.text) for p in lib.entries]
    rows = []
    for case in cases:
        wanted = terms(case["query"])
        scores = [(len(wanted & s), i) for i, s in enumerate(simple) if wanted & s]
        scores.sort(key=lambda item: (-item[0], item[1]))
        result = {"query": case["query"], "answerable": bool(case["lines"])}
        methods = {"prefix": [lib.entries[i] for _, i in scores[:3]], "bm25": lib.search(case["query"])}
        for method, hits in methods.items():
            rank = next((k for k, p in enumerate(hits, 1) if p.book == case["book"] and any(
                p.first_line <= end and start <= p.last_line for start, end in case["lines"])), None)
            result[method] = {"rank": rank, "hits": [p.ref for p in hits]}
        rows.append(result)
    positive = [r for r in rows if r["answerable"]]
    negative = [r for r in rows if not r["answerable"]]
    metrics = {}
    for name in ("prefix", "bm25"):
        ranks = [r[name]["rank"] for r in positive]
        metrics[name] = {"top1": ranks.count(1), "top3": sum(r is not None for r in ranks),
                         "mrr3": sum(1/r for r in ranks if r)/len(positive),
                         "empty": sum(not r[name]["hits"] for r in negative)}
    return {"positive": len(positive), "negative": len(negative), "scores": metrics, "queries": rows}


def alternatives(story: Scenario, votes: dict[str, str]) -> dict:
    """Минимальное расстояние до каждого другого финала, в числе изменённых бюллетеней."""
    actual = resolve(story, votes)[0]
    actors = list(story.cast)
    options = ["abstain", "challenge", *(f"select:{actor}" for actor in actors)]
    best = {}
    for values in product(options, repeat=len(actors)):
        candidate = dict(zip(actors, values))
        outcome = resolve(story, candidate)[0]
        if outcome == actual:
            continue
        changes = {key: vote for key, vote in candidate.items() if votes[key] != vote}
        distance = len(changes)
        if outcome not in best or distance < best[outcome]["distance"]:
            best[outcome] = {"distance": distance, "ways": 0, "example": changes}
        if distance == best[outcome]["distance"]:
            best[outcome]["ways"] += 1
    return best


def check_archive(events: list[dict], lib: Library) -> list[dict]:
    """Проверяем адрес, текст, автора, порядок открытия и связь с реальным ходом."""
    if not events or events[-1]["event"] != "ending":
        raise ValueError("Нужен завершённый журнал спектакля")
    selected, public, turns = {}, {}, {}
    for e in events:
        kind = e["event"]
        if kind == "turn":
            turns[e["index"]] = e
        if kind == "book_withdrawn":
            selected.pop((e["index"], e["actor"]), None)
        if kind not in {"book_search", "book_context", "citation"}:
            continue
        if events[0].get("library_revision") != lib.revision:
            raise ValueError("Журнал относится к другой редакции корпуса")
        key = (e["index"], e["actor"])
        if kind == "citation":
            if (e["index"] in public or e["index"] in turns
                    or selected.get(key) != (e["kind"], e["card"], e.get("origin"), e.get("origin_turn"))):
                raise ValueError("Для публичной цитаты нет соответствующего выбора актёра")
            public[e["index"]] = e
            continue
        cards = e["hits"]
        for card in cards:
            p = lib.get(card["ref"])
            identity = lib.card(p)
            stable = ("ref", "book", "author", "title", "speaker", "act", "section", "lines", "url")
            quote = card["quote"]
            if (any(identity[k] != card.get(k) for k in stable) or not quote or quote not in p.text
                    or not 1 <= len(quote.split()) <= 20
                    or "cut_before" in card and card["cut_before"] != (not p.text.startswith(quote))
                    or "cut_after" in card and card["cut_after"] != (not p.text.endswith(quote))):
                raise ValueError(f"Искажена цитата или адрес фрагмента {card['ref']}")
        if kind == "book_context":
            prior = public.get(e["origin_turn"])
            if (not prior or e["origin_turn"] not in turns or e["origin_turn"] >= e["index"] or prior["actor"] == e["actor"]
                    or prior["card"]["ref"] != e["origin"] or prior["scene"] != e["scene"]):
                raise ValueError("Отсутствует ранее опубликованная цитата другого актёра")
            if [c["ref"] for c in cards] != [p.ref for p in lib.context(e["origin"], radius=3)]:
                raise ValueError("Открыт посторонний контекст")
            if e["selected"] == e["origin"]:
                raise ValueError("Для проверки нужна соседняя строка")
        elif any(c["book"] != e["book"] for c in cards):
            raise ValueError("Поиск вернул другую книгу")
        if e["selected"]:
            matches = [c for c in cards if c["ref"] == e["selected"]]
            if len(matches) != 1:
                raise ValueError("Выбран отсутствующий или неоднозначный фрагмент")
            selected[key] = ("read" if kind == "book_search" else "check", matches[0],
                             e.get("origin"), e.get("origin_turn"))
    for index, citation in public.items():
        turn = turns.get(index, {})
        if any(turn.get(field) != citation[field] for field in ("actor", "scene", "name")):
            raise ValueError("Цитата не связана с принятым ходом")
    return list(public.values())


def compare_runs(control: list[dict], with_books: list[dict], lib: Library) -> dict:
    """Парный опыт требует совпадения актёров, плана и условий генерации."""
    if not control or not with_books or any(run[-1]["event"] != "ending" for run in (control, with_books)):
        raise ValueError("Для сравнения нужны два завершённых журнала.")
    settings = ("scenario", "cast", "models", "seed", "editor", "planner", "turns_per_scene")
    if any(control[0].get(k) != with_books[0].get(k) for k in settings):
        raise ValueError("Различаются условия постановок.")
    plans = [next((e["scenes"] for e in run if e["event"] == "plan"), None) for run in (control, with_books)]
    if plans[0] is None or plans[0] != plans[1]:
        raise ValueError("Для парного опыта нужен один макроплан.")
    if control[0].get("library_revision") or with_books[0].get("library_revision") != lib.revision:
        raise ValueError("Первый журнал должен быть контрольным, второй — с действующей библиотекой.")
    story = load_scenario(control[0]["scenario"])
    turns = [[e for e in run if e["event"] == "turn"] for run in (control, with_books)]
    common = 0
    for a, b in zip(*turns):
        if (a["actor"], a["line"]) != (b["actor"], b["line"]):
            break
        common += 1
    output = {"scenario": story.key, "seed": control[0]["seed"], "same_opening_turns": common}
    for key, run, accepted in zip(("control", "with_books"), (control, with_books), turns, strict=True):
        votes = run[-1]["votes"]
        output[key] = {"turns": len(accepted), "citations": len(check_archive(run, lib)),
                       "outcome": run[-1]["outcome"],
                       "votes_to_other_ending": min(v["distance"] for v in alternatives(story, votes).values())}
    a, b = control[-1]["votes"], with_books[-1]["votes"]
    output["changed_votes"] = {story.cast[k].name: [ballot_phrase(story, k, a[k]), ballot_phrase(story, k, b[k])]
                               for k in story.cast if a[k] != b[k]}
    return output


def dossier(events: list[dict], lib: Library) -> str:
    citations = check_archive(events, lib)
    story = load_scenario(events[0]["scenario"])
    turns = {e["index"]: e for e in events if e["event"] == "turn"}
    searches = {(e["index"], e["selected"]): e for e in events if e["event"] == "book_search" and e["selected"]}
    lines = [f"# Свидетельства в спектакле «{story.title}»", "", f"Зерно прогона {events[0]['seed']}. "
             f"Принятых реплик {len(turns)}, прочитанных фрагментов {len(citations)}.", ""]
    for e in citations:
        card = e["card"]
        lines.extend([f"## Ход {e['index']}. {e['name']}", ""])
        if e["kind"] == "read":
            search = searches[e["index"], card["ref"]]
            lines.append(f"Запрос актёра — «{search['query']}». Из {len(search['hits'])} предложенных фрагментов выбран `{card['ref']}`.")
        else:
            lines.append(f"Актёр открывает контекст фрагмента `{e['origin']}`, прозвучавшего на ходу {e['origin_turn']}.")
        lines.extend(["", f"Вопрос на сцене — {e['question']}", "",
                      f"Выбранное актёром намерение — {e['stance']}.", "",
                      f"> {card['speaker']}. {printed_quote(card)}", "",
                      f"[{card['author']}, «{card['title']}», действие {card['act']}, {card['section'].lower()}]({card['url']})",
                      "", f"**{e['name']}.** {turns[e['index']]['line']}", ""])
    withdrawn = [e for e in events if e["event"] == "book_withdrawn"]
    if withdrawn:
        lines.extend(["## Где актёр отложил книгу", ""])
        for e in withdrawn:
            lines.append(f"- Ход {e['index']}, {e['name']}. Фрагмент `{e['ref']}` остался за кулисами. "
                         f"Последнее замечание режиссёра — «{e['reason']}»")
        lines.append("")
    ending = events[-1]
    lines.extend(["## Действительный финал", "", ending["curtain"], ""])
    for a, v in ending["votes"].items():
        lines.append(f"- {story.cast[a].name} — {ballot_phrase(story, a, v)}")
    possibilities = alternatives(story, ending["votes"])
    distance = min(item["distance"] for item in possibilities.values())
    lines.extend(["", "## Ближайшая альтернативная развязка", "",
                  f"Минимальное число изменений бюллетеней — {distance}. Это пересчёт тех же правил после спектакля.", ""])
    for outcome, possibility in possibilities.items():
        if possibility["distance"] != distance:
            continue
        edits = possibility["example"]
        _, curtain = resolve(story, ending["votes"] | edits)
        who = " ".join(f"{story.cast[k].name}: «{ballot_phrase(story, k, v)}»." for k, v in edits.items())
        lines.append(f"- {who} Тогда: {curtain} Минимальных вариантов для этого исхода: {possibility['ways']}.")
    return "\n".join(lines) + "\n"


In [15]:
%%theatre_module cli
import argparse
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import httpx

from .book_scene import citation_intro, printed_quote
from .engine import perform
from .evidence import dossier
from .library import Library
from .llm import DEFAULT_A, DEFAULT_B, Gateway, GenerationError
from .scenario import SCENARIOS, load_scenario
from .schema import Plan


def read_saved_plan(directory: Path, scenario: str) -> tuple[Plan, str]:
    """Берём только первые два события: актёры не получат диалог прежнего состава."""
    try:
        with (directory / "trace.jsonl").open(encoding="utf-8") as journal:
            start, event = json.loads(journal.readline()), json.loads(journal.readline())
        if start["event"] != "start" or start["scenario"] != scenario:
            raise ValueError("папка относится к другой постановке")
        if event["event"] != "plan" or not isinstance(event["model"], str) or not event["model"]:
            raise ValueError("отсутствует план с именем драматурга")
        return Plan(scenes=event["scenes"]), event["model"]
    except (KeyError, TypeError, OSError, ValueError) as exc:
        raise ValueError(f"Не удалось прочитать план из {directory}: {exc}") from exc


def markdown(title: str, events: list[dict]) -> str:
    lines = [f"# {title}", ""]
    ballot_labels = {"challenge": "оспорить условие", "abstain": "воздержаться"}
    ballot_labels.update({f"select:{e['actor']}": e["name"] for e in events if e["event"] == "turn"})
    for e in events:
        if e["event"] == "scene":
            lines.extend([f"## Сцена {e['scene']}. {e['title']}", "", f"*{e['opening']}*", ""])
            if e.get("action"):
                lines.extend([f"*{e['action']}*", ""])
            if e["notice"]:
                lines.extend([f"**Табло.** {e['notice']}", ""])
        elif e["event"] == "citation":
            card = e["card"]
            lines.extend([f"*{citation_intro(e)}*", "", f"> {printed_quote(card)}", "",
                          f"[Отрывок в издании]({card['url']})", ""])
        elif e["event"] == "turn":
            if e["stage"]:
                lines.extend([f"*{e['stage']}*", ""])
            lines.extend([f"**{e['name']}.** {e['line']}", ""])
            if e["vote"]:
                lines.extend([f"*На табло у имени «{e['name']}» — «{ballot_labels[e['vote']]}».*", ""])
        elif e["event"] == "ending":
            lines.extend([f"*{e['curtain']}*", ""])
    return "\n".join(lines)


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Сыграть многоагентную пьесу на русском языке")
    parser.add_argument("scenario", choices=SCENARIOS, help="extra — Лишний персонаж, lab404 — Лабораторная № 404")
    parser.add_argument("--model-a", default=DEFAULT_A, help="модель первого и третьего актёра")
    parser.add_argument("--model-b", default=DEFAULT_B, help="модель второго и четвёртого актёра")
    planning = parser.add_mutually_exclusive_group()
    planning.add_argument("--planner", help="по умолчанию model-a; для GPT укажите openai/ID")
    planning.add_argument("--plan-from", type=Path, help="папка прежнего прогона этой пьесы, чей план сыграть заново")
    editing = parser.add_mutually_exclusive_group()
    editing.add_argument("--editor", help="модель для смысловой проверки черновика (по умолчанию model-a)")
    editing.add_argument("--no-editor", action="store_true", help="оставить только проверки Python")
    parser.add_argument("--no-library", action="store_true", help="контрольная постановка без чтения больших пьес")
    parser.add_argument("--ollama-url", default=os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434"))
    parser.add_argument("--turns", type=int, choices=range(4, 11), default=6,
                        help="ходов в каждой из первых трёх сцен, в финале всегда четыре")
    parser.add_argument("--seed", type=int, default=404)
    parser.add_argument("--out", type=Path, help="новая папка прогона (по умолчанию runs/...)")
    args = parser.parse_args(argv)
    story = load_scenario(args.scenario)
    if args.model_a == args.model_b:
        parser.error("укажите две разные модели")
    planner = args.planner or args.model_a
    plan = None
    if args.plan_from:
        try:
            plan, planner = read_saved_plan(args.plan_from, story.key)
        except ValueError as exc:
            parser.error(str(exc))
    models = {key: (args.model_a, args.model_b)[i % 2] for i, key in enumerate(story.cast)}
    editor = None if args.no_editor else args.editor or args.model_a
    library = None if args.no_library else Library()
    out = args.out or Path("runs") / f"{story.key}-{datetime.now():%Y%m%d-%H%M%S-%f}"
    try:
        out.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        parser.error(f"папка {out} уже существует; укажите новую, чтобы сохранить предыдущий прогон")

    events: list[dict] = []
    url = args.ollama_url if "://" in args.ollama_url else "http://" + args.ollama_url
    with (out / "trace.jsonl").open("x", encoding="utf-8") as journal:
        def emit(e: dict) -> None:
            journal.write(json.dumps(e, ensure_ascii=False) + "\n")
            journal.flush()
            events.append(e)
            if e["event"] == "plan":
                (out / "plan.json").write_text(json.dumps(e, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            elif e["event"] == "scene":
                print(f"{e['scene']}. {e['title']}", flush=True)

        emit({"event": "start", "scenario": story.key, "seed": args.seed,
              "models": models, "planner": planner, "turns_per_scene": args.turns,
              "library_revision": library.revision if library else None,
              "cast": {key: {"name": a.name, "gender": a.gender, "speaker": a.speaker}
                       for key, a in story.cast.items()},
              "plan_from": str(args.plan_from) if args.plan_from else None, "editor": editor})
        with httpx.Client(timeout=httpx.Timeout(180, connect=10)) as http:
            try:
                perform(story, Gateway(http, url, args.seed), models, planner, args.turns, emit,
                        plan=plan, editor=editor, library=library)
            except GenerationError as exc:
                emit({"event": "error", "message": str(exc)})
                print(f"Прогон прерван. Журнал: {out}\n{exc}", file=sys.stderr)
                return 1
    (out / "play.md").write_text(markdown(story.title, events), encoding="utf-8")
    if library:
        (out / "dossier.md").write_text(dossier(events, library), encoding="utf-8")
    print(f"Готово: {out / 'play.md'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [16]:
%%theatre_module reading
"""Читка по актам. Импровизация, вызовы инструментов и голоса остаются в исходном журнале."""

import argparse
import hashlib
import json
import os
from pathlib import Path
import re
from tempfile import NamedTemporaryFile

import httpx
from pydantic import Field, create_model

from .analysis import read_trace
from .book_scene import acting_note, check_comment
from .engine import ballot_phrase
from .language import stage_direction
from .llm import Gateway, GenerationError, Generator
from .rehearsal import draft_issues
from .scenario import Scenario, load_scenario
from .schema import Delivery, Form


class RoleLine(Form):
    line: str = Field(alias="реплика")
    stage: str = Field(alias="жест")


class ReadingLine(RoleLine):
    index: int


class SceneReading(Form):
    turns: list[ReadingLine]


# Темы реплик, которые можно перенести в читку без выдуманного факта из черновика.
TOPICS = {
    "семья и цена возвращения": "моник|дет(?:ей|и|ям)|семь|домой",
    "чья память может служить доказательством": "памят|помн|воспомина",
    "можно ли отказаться от навязанного выбора": "отказ|согласи|воздерж|навяз",
    "что на самом деле измеряет прибор": "датчик|измер|фиксир|показан",
    "кто имеет право судить участников": "наблюдател|судить|подпис|критер",
    "условия совместной попытки": "вместе|сообща|четвером|пересмотр",
    "что проверено у двери": "ручк|замок|замка|двер",
    "возможная проверка предложения": "провер|опыт|гипотез",
}


def scene_take(story: Scenario, turns: list[dict], take: SceneReading,
               citations: dict[int, dict] | None = None) -> list[dict]:
    if [line.index for line in take.turns] != [turn["index"] for turn in turns]:
        raise ValueError("Сохрани точное число, порядок и номера реплик.")
    result = []
    for turn, words in zip(turns, take.turns, strict=True):
        actor = story.cast[turn["actor"]]
        staying = story.key == "extra" and turn["vote"] == "select:" + turn["actor"]
        pledge = ballot_phrase(story, turn["actor"], turn["vote"]) if turn["vote"] else ""
        names = "|".join(re.escape(n) for n in [actor.name, actor.name.split()[0]])
        line = re.sub(rf"^\s*(?:{turn['index']}[.)]\s*)?(?:{names})\s*[:.]\s*", "", words.line)
        line = line.removeprefix("—").strip()
        if line.startswith("«") and line.endswith("»"):
            line = line[1:-1].strip()
        if re.match(r"^\d+[.)]\s|^[А-ЯЁ][а-яё]+(?: [А-ЯЁ][а-яё]+)?:\s", line):
            raise ValueError(f"В поле реплики {turn['index']} должны быть только слова этого актёра.")
        if pledge:
            options = ["challenge", "abstain", *(f"select:{k}" for k in story.cast)]
            if any(other != pledge and line.startswith(other) for other in
                   (ballot_phrase(story, turn["actor"], option) for option in options)):
                raise ValueError(f"Ход {turn['index']}: решение уже было зарегистрировано как «{pledge}».")
        line = line.removeprefix(pledge).strip() if pledge else line
        clean = Delivery(line=line, stage=words.stage.strip("()* "))
        if citation := (citations or {}).get(turn["index"]):
            cast_text = " ".join(a.name + " " + a.stake for a in story.cast.values())
            verdict = check_comment(clean, citation, cast_text)
            if verdict.issue:
                raise ValueError(f"Ход {turn['index']}: {verdict.note}")
        issues = draft_issues(clean.model_copy(update={"stage": ""}), actor, [], staying,
                              partners=list(story.cast.values()), addressee=story.cast.get(turn["to"]))
        if issues:
            raise ValueError(f"Ход {turn['index']} ({actor.name}, {actor.agreement}): {', '.join(issues)}.")
        gesture = clean.stage
        stage = stage_direction(gesture, actor) if len(gesture.split()) <= 12 else ""
        result.append({"index": words.index, "line": (pledge + " " + line).strip(), "stage": stage})
    return result


def apply_reading(events: list[dict], reading: list[dict]) -> list[dict]:
    turns = [e for e in events if e["event"] == "turn"]
    if [t["index"] for t in turns] != [r["index"] for r in reading]:
        raise ValueError("Номера реплик читки не совпадают с журналом.")
    replacements = iter(reading)
    return [e | {k: v for k, v in next(replacements).items() if k in {"line", "stage"}}
            if e["event"] == "turn" else e for e in events]


def read_performance(run: Path, *, edition: bool = True) -> list[dict]:
    events = read_trace(run)
    file = run / "reading.json"
    if not file.is_file():
        return events
    reading = json.loads(file.read_text(encoding="utf-8"))
    if reading["trace_sha256"] != hashlib.sha256((run / "trace.jsonl").read_bytes()).hexdigest():
        raise ValueError("Журнал изменился после читки. Для записи нужен соответствующий ему текст.")
    performance = apply_reading(events, reading["turns"])
    edited = run / "edition.json"
    if edition and edited.is_file():
        changes = json.loads(edited.read_text(encoding="utf-8"))
        if changes["reading_sha256"] != hashlib.sha256(file.read_bytes()).hexdigest():
            raise ValueError("Финальная редактура относится к другой читке.")
        return revise_lines(performance, changes["changes"])
    return performance


def revise_lines(events: list[dict], changes: dict[str, dict]) -> list[dict]:
    story = load_scenario(events[0]["scenario"])
    turns = [e for e in events if e["event"] == "turn"]
    if not isinstance(changes, dict) or not set(changes) <= {str(t["index"]) for t in turns}:
        raise ValueError("Исправить можно только существующий ход читки.")
    revised = []
    citations = {e["index"]: e for e in events if e["event"] == "citation"}
    for turn in turns:
        edited = changes.get(str(turn["index"]))
        if edited is None:
            revised.append({key: turn[key] for key in ("index", "line", "stage")})
            continue
        if not edited or set(edited) - {"line", "stage"}:
            raise ValueError("В правке допустимы только поля line и stage.")
        text = ReadingLine(index=turn["index"], реплика=edited.get("line", turn["line"]),
                           жест=edited.get("stage", turn["stage"]))
        revised.extend(scene_take(story, [turn], SceneReading(turns=[text]), citations))
    return apply_reading(events, revised)


def publish_edition(run: Path, changes: dict, credit: str) -> None:
    from .cli import markdown

    reading = run / "reading.json"
    digest = hashlib.sha256(reading.read_bytes()).hexdigest()
    events = revise_lines(read_performance(run, edition=False), changes)
    payload = {"editor": credit, "reading_sha256": digest, "changes": changes}
    publish_pair(run / "edition.md", markdown(load_scenario(events[0]["scenario"]).title, events),
                 run / "edition.json", payload)


def rehearse_scene(story: Scenario, scene: dict, turns: list[dict], available: dict,
                   previous: list[str], gen: Generator, model: str,
                   notes: dict[str, str] | None = None, citations: dict[int, dict] | None = None,
                   kept_turns: dict[int, str] | None = None) -> list[dict]:
    cast = "\n".join(f"{a.name} ({a.agreement}): {a.voice} Известно: {a.stake}" for a in story.cast.values())
    lines = []
    for i, t in enumerate(turns):
        ref = (f" Опора: {available[t['actor'], t['source']]}" if t["source"] else "")
        vote = ballot_phrase(story, t["actor"], t["vote"]) if t["vote"] else ""
        themes = [label for label, pattern in TOPICS.items() if re.search(pattern, t["line"], re.I)]
        following = f" Затем говорит {turns[i+1]['name']}." if i+1 < len(turns) else " Конец действия."
        lines.append(f"{t['index']}. {t['name']}. Мотивы её или его импровизации: "
                     f"{', '.join(themes[:2]) or 'реакция на партнёра'}. "
                     f"Отвечает на ход {t.get('reply_to') or 'открытие сцены'}.{ref}"
                     + (f" Обязательный голос: {vote}" if vote else "") + following)
        if direction := (notes or {}).get(str(t["index"])):
            lines.append("Режиссёрское задание для этой реплики: " + direction)
    system = ("Ты драматург русской камерной пьесы. Актёры уже сыграли и приняли решения. "
              "Сделай устную читку: четыре разных живых человека слушают друг друга. Коротко, по-русски. "
              "Произнеси ответ на конкретный вопрос. Один собеседник вправе возразить, уступить или "
              "признать незнание. Если другой герой верно его понял, не начинай ответ с пересказа. "
              "Береги сухой юмор Чендлера, точность Гермионы, прямоту Барби. "
              "Пример ритма (не копируй): «Сколько ты теряешь, если остаёшься?» — «Ужин дома. "
              "Смешно звучит, пока не представишь пустую тарелку». — «Тогда почему ты отодвинул стул?» "
              "Каждая фраза обращена к конкретному человеку. Дай разговору начаться, измениться и закончиться.")
    canon = ("Терминал гарантирует три пропуска. У него нет измерения ценности людей. Тони не управляет "
             "магией Гермионы. Четвёртый человек может не получить пропуск."
             if story.key == "extra" else
             "Терминал не знает, кто именно настоящий, и никому не присвоил этот статус. На экране только "
             "условие задачи. Никто из этих четверых до этой встречи не знал другого лично.")
    if scene["scene"] == 1:
        canon = ("Четверо встретились впервые. Дверь существует и остаётся закрытой после проверки. "
                 "Экран тёмный. О выборе, голосовании и пропусках никто ещё не слышал. "
                 "Начни с обычного разговора: кому нужно вернуться, что стоящий рядом может сделать для него.")
    direction = " ".join(story.cues[scene["scene"]-1])
    prompt = (f"«{story.title}». {story.setting}\nВерные условия мира: {canon}\nРоли:\n{cast}\n"
              "Уже прозвучало до этого действия:\n"
              + ("\n".join(previous[-5:]) or "(пьеса только началась)")
              + f"\nСцена {scene['scene']}. {story.arc[scene['scene']-1]}\n"
                f"На сцене: {scene['opening']} {scene.get('action', '')}\n"
              + (f"Объявление: {story.notice}\n" if scene["scene"] >= 2 else "")
              + f"Задача драматурга в этом действии: {direction}\nХоды актёров в фактическом порядке:\n" + "\n".join(lines) +
              "\nЖёсткие условия — говорящие, их решения, правила терминала, полученные улики и мизансцена. "
              "Свободно перепиши неудачные доводы. Если в черновике путаются причины и следствия, "
              "придумывают измерения, чужие способности или тянут случайную метафору, выброси это целиком. "
              "Опирайся на наблюдаемое действие, личный интерес конкретного героя и услышанный вопрос. "
              "Если герой предлагает измерить неизмеримое, пусть другой спросит, как именно, и получит ответ. "
              "Исправь ложные цитаты и несовместимые утверждения. Гипотезы остаются гипотезами. "
              "Если актёр отвечает за другого, верни каждому его собственную мысль. Никаких новых "
              "событий прошлого, родственников, чисел датчика, надписей на экране или открытий двери. "
              "Проверку ручки уже видели. Помни, кто выдвинул какой довод. В каждой строке точный index, "
              "Избегай отвлечённых метафор и загадок. Ручка, стул, семейный ужин, пустая строка "
              "эксперимента — это буквальные предметы и обстоятельства. Если задаёшь прямой вопрос, дай ответить тому, кто действительно говорит следующим. "
              "Обращение «ты» и род согласуй с ним. У персонажей нет общих приключений в прошлом. "
              "одна-две законченные произносимые фразы (до 32 слов) и необязательный короткий жест "
              "от третьего лица. В реплике только произносимые слова, без номера и имени. В финале пиши только обоснование, "
              "формулу голоса программа прикрепит сама. У актёров не может быть чужих реплик.")
    # Идентичность говорящего принадлежит журналу. Отдельное обязательное поле
    # на каждый ход исключает перепутанные номера и пропущенные реплики в JSON.
    keys = [f"turn_{turn['index']}_{turn['actor']}" for turn in turns]
    sheet_type = create_model("ActScript", __base__=Form, **{key: (RoleLine, ...) for key in keys})
    prompt += "\nИмена полей в порядке исполнения: " + ", ".join(keys) + "."
    correction = ""
    for attempt in range(3):
        sheet = gen.ask(model, system, prompt + correction, sheet_type, budget=max(850, len(turns) * 190))
        take = SceneReading(turns=[ReadingLine(index=t["index"], **getattr(sheet, k).model_dump(by_alias=True))
                                   for t, k in zip(turns, keys, strict=True)])
        try:
            return scene_take(story, turns, take, citations)
        except ValueError as exc:
            if attempt == 2:
                # Когда монтаж портит книжную реплику, можно взять уже сыгранный
                # дубль. Он проходит те же проверки; происхождение сохраняем.
                final = []
                for t, line in zip(turns, take.turns, strict=True):
                    try:
                        segment = scene_take(story, [t], SceneReading(turns=[line]), citations)
                    except ValueError as mistake:
                        if t["index"] not in (citations or {}):
                            raise GenerationError(f"Не принята читка сцены {scene['scene']}: {mistake}") from mistake
                        original = ReadingLine(index=t["index"], реплика=t["line"], жест=t["stage"])
                        try:
                            segment = scene_take(story, [t], SceneReading(turns=[original]), citations)
                        except ValueError as invalid:
                            raise GenerationError(f"Не принят и исходный дубль: {invalid}") from invalid
                        if kept_turns is not None:
                            kept_turns[t["index"]] = str(mistake)
                    final.extend(segment)
                return final
            correction = f"\nИсправь предыдущую попытку и верни все строки сцены. {exc}"
    raise AssertionError("unreachable")


def prepare_reading(run: Path, gen: Generator, model: str) -> None:
    from .cli import markdown

    events = read_trace(run)
    if not events or events[-1]["event"] != "ending":
        raise ValueError("Для читки нужен завершённый спектакль.")
    manifest, script = run / "reading.json", run / "reading.md"
    if manifest.exists() or script.exists():
        raise FileExistsError("В папке уже есть читка.")
    story = load_scenario(events[0]["scenario"])
    direction_file = run / "directions.json"
    notes = json.loads(direction_file.read_text(encoding="utf-8")) if direction_file.exists() else {}
    indices = {str(e["index"]) for e in events if e["event"] == "turn"}
    if not isinstance(notes, dict) or not set(notes) <= indices or any(
            not isinstance(text, str) or not text.strip() for text in notes.values()):
        raise ValueError("В режиссёрском листе ожидаются номера сыгранных ходов и непустые задания.")
    available = {(e["actor"], f"{h['file']}:{h['line']}"): h["text"]
                 for e in events if e["event"] == "search" for h in e["hits"]}
    scene_notes = dict(notes)
    citations = {e["index"]: e for e in events if e["event"] == "citation"}
    for index, event in citations.items():
        key = str(index)
        scene_notes[key] = (scene_notes.get(key, "") + " " + acting_note(event)).strip()
    previous, reading, kept_turns = [], [], {}
    for scene in (e for e in events if e["event"] == "scene"):
        turns = [e for e in events if e["event"] == "turn" and e["scene"] == scene["scene"]]
        new = rehearse_scene(story, scene, turns, available, previous, gen, model, scene_notes, citations, kept_turns)
        reading.extend(new)
        previous.extend(f"{t['name']}: {v['line']}" for t, v in zip(turns, new, strict=True))
    data = {"model": model, "trace_sha256": hashlib.sha256((run / "trace.jsonl").read_bytes()).hexdigest(),
            "turns": reading}
    if notes:
        data["directions_sha256"] = hashlib.sha256(direction_file.read_bytes()).hexdigest()
    if kept_turns:
        data["retained_from_improvisation"] = kept_turns
    publish_pair(script, markdown(story.title, apply_reading(events, reading)), manifest, data)


def publish_pair(script: Path, text: str, manifest: Path, data: dict) -> None:
    # Публикуем только полную читку. Повторный процесс не заменяет ранее созданные файлы.
    with NamedTemporaryFile(dir=script.parent, mode="w", encoding="utf-8") as sheet, \
            NamedTemporaryFile(dir=manifest.parent, mode="w", encoding="utf-8") as journal:
        sheet.write(text)
        journal.write(json.dumps(data, ensure_ascii=False, indent=2) + "\n")
        sheet.flush()
        journal.flush()
        os.link(sheet.name, script)
        try:
            os.link(journal.name, manifest)
        except OSError:
            script.unlink()
            raise


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Провести литературную читку завершённой импровизации")
    parser.add_argument("run", type=Path)
    parser.add_argument("--model", default="gpt-oss:20b", help="литературный редактор")
    parser.add_argument("--edits", type=Path, help="применить JSON правок к существующей читке, без LLM")
    parser.add_argument("--credit", default="Редактор постановки", help="автор финальной вычитки")
    parser.add_argument("--ollama-url", default=os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434"))
    args = parser.parse_args(argv)
    try:
        if args.edits:
            publish_edition(args.run, json.loads(args.edits.read_text(encoding="utf-8")), args.credit)
            print(f"Готова редакция: {args.run / 'edition.md'}")
            return 0
        start = read_trace(args.run)[0]
        url = args.ollama_url if "://" in args.ollama_url else "http://" + args.ollama_url
        with httpx.Client(timeout=httpx.Timeout(240, connect=10)) as http:
            prepare_reading(args.run, Gateway(http, url, start["seed"]), args.model)
    except (GenerationError, OSError, ValueError) as exc:
        parser.exit(1, f"Читка не завершена. {exc}\n")
    print(f"Читка готова: {args.run / 'reading.md'}")
    return 0


In [17]:
%%theatre_module audio
"""Озвучка готового журнала. Для чтения и генерации пьесы PyTorch не требуется."""

import argparse
from collections.abc import Iterator
from dataclasses import dataclass
import hashlib
from html import escape
import os
from pathlib import Path
import re
from tempfile import NamedTemporaryFile
import wave

from .reading import read_performance
from .book_scene import citation_intro, printed_quote
from .language import stage_direction
from .scenario import NARRATOR, VoiceRole, load_cast, load_scenario


MODEL_URL = "https://models.silero.ai/models/tts/ru/v5_6_ru.pt"
MODEL_SHA256 = "04cc9b1d08ce38bcaa9d02c488d80d3e6806d17f7eaa7c95982f36a3cbf50b2e"
LEGACY_SHA256 = "896ab96347d5bd781ab97959d4fd6885620e5aab52405d3445626eb7c1414b00"
VOICES = {key: actor.speaker for key, actor in load_cast().items()}
RATE = 48_000


@dataclass(frozen=True)
class Cue:
    speaker: str
    name: str
    text: str
    pause_ms: int


def casting(events: list[dict]) -> dict[str, VoiceRole]:
    start = next((e for e in events if e["event"] == "start"), {})
    if "cast" in start:
        roles = {key: VoiceRole(**card) for key, card in start["cast"].items()}
        voices = [NARRATOR, *(role.speaker for role in roles.values())]
        if len(voices) != len(set(voices)):
            raise ValueError("в журнале один голос назначен нескольким участникам")
        return roles
    return load_scenario(start["scenario"]).cast if "scenario" in start else load_cast()


def cues_for(events: list[dict]) -> Iterator[Cue]:
    """Первая реплика знакомит слушателя с голосом, ремарки звучат у рассказчицы."""
    if not events or events[-1].get("event") != "ending":
        raise ValueError("для озвучки нужен завершённый спектакль с развязкой")
    roles = casting(events)
    introduced = set()
    ordinals = ("первая", "вторая", "третья", "четвёртая")
    for e in events:
        if e["event"] == "scene":
            yield Cue(NARRATOR, "Рассказчица", f"Сцена {ordinals[e['scene'] - 1]}. {e['title']}.", 450)
            yield Cue(NARRATOR, "Рассказчица", e["opening"], 380)
            if e.get("action"):
                yield Cue(NARRATOR, "Рассказчица", e["action"], 380)
            if e["notice"]:
                yield Cue(NARRATOR, "Табло", "На табло. " + e["notice"], 550)
        elif e["event"] == "citation":
            role = roles[e["actor"]]
            yield Cue(NARRATOR, "Рассказчица", citation_intro(e), 180)
            yield Cue(role.speaker, role.name + " читает", printed_quote(e["card"]), 320)
        elif e["event"] == "turn":
            role = roles[e["actor"]]
            if e["actor"] not in introduced:
                yield Cue(NARRATOR, "Рассказчица", role.name + ".", 180)
                introduced.add(e["actor"])
            if e["stage"]:
                yield Cue(NARRATOR, "Рассказчица", stage_direction(e["stage"], role), 180)
            yield Cue(role.speaker, role.name, e["line"], 220 if e["line"].rstrip().endswith("?") else 330)
        elif e["event"] == "ending":
            yield Cue(NARRATOR, "Рассказчица", e["curtain"], 550)


def score(events: list[dict]) -> Iterator[tuple[str, str]]:
    for cue in cues_for(events):
        yield cue.speaker, cue.text


def spoken_text(text: str) -> str:
    text = re.sub(r"[«»_]", "", text.replace("**", "*"))
    for name, stressed in (("Гермион", "Герми+он"), ("Чендлер", "Ч+ендлер"), ("Барби", "Б+арби")):
        text = re.sub(r"\b" + name, stressed, text, flags=re.I)
    return text.replace("№ 404", "номер четыреста четыре").strip()


def master(clip, narrator: bool):
    import numpy as np

    sound = clip.detach().cpu().numpy().astype(np.float32, copy=True).reshape(-1)
    if not len(sound) or not np.isfinite(sound).all():
        raise ValueError("синтезатор вернул пустой или повреждённый звук")
    peak = float(np.max(np.abs(sound)))
    voiced = np.flatnonzero(np.abs(sound) > max(0.001, peak * 0.008))
    if not len(voiced):
        raise ValueError("синтезатор вернул тишину вместо реплики")
    margin = RATE // 30
    sound = sound[max(0, voiced[0]-margin):voiced[-1]+margin+1]
    active = sound[np.abs(sound) > peak * 0.06]
    target = 0.13 if narrator else 0.15
    gain = min(0.90 / peak, float(np.clip(target / np.sqrt(np.mean(active**2)), 0.65, 1.65)))
    sound *= gain
    fade = min(RATE // 200, len(sound) // 2)
    if fade:
        ramp = np.linspace(0, 1, fade, dtype=np.float32)
        sound[:fade] *= ramp
        sound[-fade:] *= ramp[::-1]
    return (sound * 32767).round().astype("<i2")


def timestamp(frame: int) -> str:
    total_ms = round(frame * 1000 / RATE)
    seconds, millis = divmod(total_ms, 1000)
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}.{millis:03}"


def load_voice_model(path: Path | None):
    import torch

    if path is None:
        path = Path(torch.hub.get_dir()) / "checkpoints" / "silero-v5-6-ru.pt"
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.hub.download_url_to_file(MODEL_URL, str(path), hash_prefix=MODEL_SHA256)
    with path.open("rb") as source:
        if hashlib.file_digest(source, "sha256").hexdigest() not in {MODEL_SHA256, LEGACY_SHA256}:
            raise ValueError(f"контрольная сумма модели Silero не совпала: {path}")
        source.seek(0)
        model = torch.package.PackageImporter(source).load_pickle("tts_models", "model")
    torch.set_num_threads(min(4, torch.get_num_threads()))
    model.to(torch.device("cpu"))
    return model


def record(events: list[dict], out: Path, model_path: Path | None = None, *, captions: Path | None = None) -> None:
    # Ни прерванный синтез, ни параллельный запуск не заменяют готовую фонограмму.
    cues = list(cues_for(events))
    if out.exists():
        raise FileExistsError(f"Фонограмма уже существует: {out}")
    if captions is not None and (captions.exists() or captions.resolve() == out.resolve()):
        raise FileExistsError(f"Для субтитров нужен отдельный новый файл: {captions}")
    import torch

    model = load_voice_model(model_path)
    subtitles = ["WEBVTT", ""]
    frame = 0
    out.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=out.parent, suffix=".wav") as partial:
        with wave.open(partial, "wb") as wav, torch.inference_mode():
            wav.setparams((1, 2, RATE, 0, "NONE", "not compressed"))
            for cue in cues:
                clip = model.apply_tts(text=spoken_text(cue.text), speaker=cue.speaker, sample_rate=RATE)
                pcm = master(clip, cue.speaker == NARRATOR)
                wav.writeframes(pcm.tobytes())
                if captions is not None:
                    subtitles.extend((f"{timestamp(frame)} --> {timestamp(frame + len(pcm))}",
                                      f"<v {escape(cue.name)}>{escape(cue.text)}</v>", ""))
                pause = round(RATE * cue.pause_ms / 1000)
                wav.writeframes(bytes(pause * 2))
                frame += len(pcm) + pause
        partial.flush()
        if captions is None:
            os.link(partial.name, out)
        else:
            captions.parent.mkdir(parents=True, exist_ok=True)
            with NamedTemporaryFile(dir=captions.parent, mode="w", encoding="utf-8") as vtt:
                vtt.write("\n".join(subtitles))
                vtt.flush()
                os.link(vtt.name, captions)
                try:
                    os.link(partial.name, out)
                except OSError:
                    captions.unlink()
                    raise


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Озвучить завершённую пьесу голосами Silero v5.6")
    parser.add_argument("run", type=Path, help="папка с trace.jsonl")
    parser.add_argument("--out", type=Path, help="новый WAV-файл, по умолчанию play.wav в папке спектакля")
    parser.add_argument("--model", type=Path, help="скачанный пакет Silero v5_6_ru.pt или прежний v4_ru.pt")
    parser.add_argument("--captions", type=Path, help="новый WebVTT с именами и точными границами реплик")
    args = parser.parse_args(argv)
    out = args.out or args.run / "play.wav"
    try:
        record(read_performance(args.run), out, args.model, captions=args.captions)
    except ModuleNotFoundError as exc:
        parser.exit(1, f"Нужны аудиозависимости: poetry install --with audio ({exc.name})\n")
    except (OSError, ValueError, RuntimeError, KeyError) as exc:
        parser.exit(1, f"Не удалось записать фонограмму: {exc}\n")
    print(f"Фонограмма готова: {out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [18]:
%%theatre_module library_cli
"""Просмотр книжной полки, проверка поиска и разбор сохранённой постановки без LLM."""
import argparse
import json
from pathlib import Path

from .analysis import read_trace
from .evidence import benchmark, compare_runs, dossier
from .library import Library


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Полнотекстовая библиотека агентского театра")
    commands = parser.add_subparsers(dest="command", required=True)
    search = commands.add_parser("search", help="найти реплики и ремарки во всём корпусе")
    search.add_argument("query")
    search.add_argument("--book", choices=("garden", "inspector", "dowry"), default="")
    search.add_argument("--limit", type=int, default=3)
    view = commands.add_parser("open", help="прочитать фрагмент и соседние реплики в том же явлении")
    view.add_argument("ref")
    view.add_argument("--radius", type=int, default=2)
    evaluation = commands.add_parser("bench", help="сравнить два метода на редакторской разметке")
    evaluation.add_argument("--json", action="store_true", help="все запросы, адреса, ранги и метрики")
    report = commands.add_parser("dossier", help="сверить с книгами цитаты спектакля и пересчитать ближайший финал")
    report.add_argument("run", type=Path)
    pair = commands.add_parser("compare", help="сопоставить два спектакля на одном плане и зерне")
    pair.add_argument("control", type=Path, help="спектакль без библиотеки")
    pair.add_argument("with_books", type=Path, help="спектакль с библиотекой")
    args = parser.parse_args(argv)
    try:
        lib = Library()
        if args.command == "dossier":
            print(dossier(read_trace(args.run), lib), end="")
        elif args.command == "compare":
            comparison = compare_runs(read_trace(args.control), read_trace(args.with_books), lib)
            print(json.dumps(comparison, ensure_ascii=False, indent=2))
        elif args.command == "bench":
            data = benchmark(lib)
            if args.json:
                print(json.dumps(data, ensure_ascii=False, indent=2))
            else:
                print(f"С ответом: {data['positive']}. Без ответа: {data['negative']}.")
                for name, m in data["scores"].items():
                    print(f"{name}: Hit@1 {m['top1']}, Hit@3 {m['top3']}, MRR@3 {m['mrr3']:.3f}, "
                          f"верных пустых выдач {m['empty']}.")
        else:
            hits = (lib.search(args.query, book=args.book, limit=args.limit) if args.command == "search"
                    else lib.context(args.ref, args.radius))
            if not hits:
                print("Совпадений не найдено.")
            for p in hits:
                print(f"{p.ref} · {lib.books[p.book].title} · действие {p.act} · {p.heading}\n"
                      f"{p.speaker}: {p.text}\n{lib.source(p)}\n")
    except (ValueError, FileNotFoundError) as exc:
        parser.exit(1, f"{exc}\n")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Постановка

В режиме live драматург пишет новый план, затем четыре актёра разыгрывают пьесу. При просмотре можно воспроизвести сохранённую постановку с теми же персонажами.

In [19]:
from theatre_notebook import engine
from theatre_notebook.analysis import read_trace
from theatre_notebook.cli import main as run_theatre, markdown
from theatre_notebook.reading import main as run_reading, read_performance
from theatre_notebook.scenario import load_scenario
from theatre_notebook.audio import record, cues_for, RATE, NARRATOR

assert Path(engine.__file__).resolve().parent == package.resolve()
story = load_scenario(SCENARIO)
reference_run = examples / "library" / SCENARIO
if MODE == "live":
    import httpx

    output_root = Path(os.environ.get("THEATRE_RUNS", str(Path.cwd() / "runs")))
    run = output_root / f"notebook-{SCENARIO}-{datetime.now():%Y%m%d-%H%M%S-%f}"
    # Получаем метаданные именно от сервера, который будет играть. Сохраняем только
    # идентификатор модели, семейство, квантование и дайджест установленных весов.
    models = {}
    with httpx.Client(base_url=OLLAMA, timeout=15) as client:
        response = client.get("/api/tags")
        response.raise_for_status()
        installed = {m["name"]: m for m in response.json()["models"]}
        for identifier in (PLANNER, MODEL_A, MODEL_B):
            if identifier not in installed:
                raise RuntimeError(f"Для генерации загрузите модель командой ollama pull {identifier}")
            model = installed[identifier]
            models[identifier] = {"digest": model["digest"], "family": model["details"]["family"],
                                  "quantization": model["details"]["quantization_level"]}
    args = [SCENARIO, "--model-a", MODEL_A, "--model-b", MODEL_B, "--planner", PLANNER,
            "--turns", str(TURNS), "--seed", str(SEED), "--out", str(run), "--ollama-url", OLLAMA]
    args += ["--editor", EDITOR] if EDITOR else ["--no-editor"]
    if run_theatre(args):
        raise RuntimeError(f"Постановка прервана. Последний принятый ход сохранён в {run / 'trace.jsonl'}")
    (run / "models.json").write_text(json.dumps(models, ensure_ascii=False, indent=2), encoding="utf-8")
    if READING_MODEL:
        run_reading([str(run), "--model", READING_MODEL, "--ollama-url", OLLAMA])
else:
    run = reference_run

events = read_trace(run)
performance = read_performance(run)
start, ending = events[0], events[-1]
turns = [e for e in events if e["event"] == "turn"]
plan = next(e for e in events if e["event"] == "plan")
print(f"{story.title} · seed {start['seed']} · {len(turns)} принятых реплик.")
print("Новая импровизация" if MODE == "live" else "Архивная импровизация и вычитанный текст для записи")
if MODE == "live":
    print("Результаты сохранены в", run)

Лишний персонаж · seed 58 · 16 принятых реплик.


Архивная импровизация и вычитанный текст для записи


## Кто играет и откуда взялся план

Модель может играть несколько персонажей. Что делает их разными, можно увидеть в карточке, образце фразы и том, что конкретный актёр рискует потерять. Русские GGUF-веса Gemma запускаются на той же машине через Ollama. [Карточка весов на Hugging Face](https://huggingface.co/unsloth/gemma-3-4b-it-GGUF)

In [20]:
print(f"Конфликт · {story.question}\n")
for key, actor in story.cast.items():
    print(f"{actor.name} · {start['models'][key]} · {actor.agreement} · голос {actor.speaker}")
    print(f"Личная ставка · {actor.stake}\nОбразец для модели · «{actor.sample}»\n")

# В контрольной паре воспроизводился один ранее написанный план. Доходим до премьеры.
parent_events = events
visited = set()
while parent_events[1].get("reused"):
    parent = parent_events[0]["plan_from"]
    assert parent not in visited
    visited.add(parent)
    parent_events = read_trace(bundle / parent)
    assert parent_events[1]["scenes"] == plan["scenes"]
original_plan = next(e for e in parent_events if e["event"] == "plan")
assert original_plan["model"] == PLANNER and not original_plan.get("reused")
assert next(i for i, e in enumerate(parent_events) if e["event"] == "plan") < next(i for i, e in enumerate(parent_events) if e["event"] == "turn")
print("Автор плана:", original_plan["model"], "· зерно премьеры", parent_events[0]["seed"])
for i, scene in enumerate(plan["scenes"], 1):
    print(f"\n{i}. {scene['title']}\nЗадача · {scene['objective']}\nОткрытие · {scene['opening']}")

Конфликт · Можно ли измерить право человека на существование его пользой для других?

Тони Старк · qwen2.5:7b · мужской род · голос eugene
Личная ставка · Однажды ты решил за всех ради их же блага. Боишься снова перепутать ответственность с правом распоряжаться чужой жизнью.
Образец для модели · «Дайте мне минуту на замок. На совесть минуты, боюсь, не хватит.»

Гермиона Грейнджер · hf.co/unsloth/gemma-3-4b-it-GGUF:Q4_K_M · женский род · голос kseniya
Личная ставка · Твои родители — дантисты, обычные люди без магии. Ты изменила их воспоминания о себе, чтобы защитить, и знаешь цену решения за близкого.
Образец для модели · «Прочти последнюю строку вслух. Где именно ты там увидел наше согласие?»

Чендлер Бинг · qwen2.5:7b · мужской род · голос aidar
Личная ставка · Дома ждут Моника и дети. Ты не умеешь спасать вселенные, но хочешь вернуться к ним и смочь объяснить свой выбор.
Образец для модели · «На собеседовании меня хотя бы спрашивали, почему я хочу здесь работать.»

Барби · hf.co/unsl

<a id="performance"></a>

## Спектакль

В архивной демонстрации показана версия, которую услышит зритель. Под ней можно сопоставить исходную импровизацию и читку. В живом запуске выступают заново сгенерированные актёры. Книжные ссылки ведут к зафиксированному изданию.

In [21]:
display(Markdown(markdown(story.title, performance)))

# Лишний персонаж

## Сцена 1. Первые попытки

*Тони, Гермиона, Чендлер и Барби входят. Тони поднимает дверную ручку — звуки стального скрежета.*

*Тони опускает дверную ручку до упора. Дверь не поддаётся. Он отступает и показывает остальным пустые ладони.*

*Осторожно разжимает пальцы.*

**Тони Старк.** Гермиона, подержи ручку, я отпущу. Защёлка возвращается на место. Вот и всё, что я пока могу вам сказать.

*Отпускает ручку.*

**Гермиона Грейнджер.** Тогда оставим дверь в покое. Чендлер, ты уже дважды посмотрел на часы. Тебя к какому-то часу ждут?

*Придерживает соседний стул.*

**Чендлер Бинг.** К ужину. Моника велела прийти вовремя хоть раз. У меня впервые есть приличное оправдание, и ни малейшего способа ей его передать.

*Садится лицом к Чендлеру.*

**Барби.** Садись рядом, Чендлер. Расскажешь про ужин, если захочешь. С головой, повёрнутой к двери, у меня тоже плохо получается ждать.

## Сцена 2. Три критерии

*Тони нажимает красную кнопку. На табло вспыхивают три зелёных черты.*

*На табло вспыхивают три зелёные черты. Четвёртый стул стоит на том же месте.*

**Табло.** Четыре человека претендуют на три гарантированных места. Панель требует выбрать того, чьё отсутствие меньше всего скажется на его мире.

*Гермиона Грейнджер открывает книгу. «Бесприданница», действие 4. Авторская ремарка.*

> (Вынимает из кармана монету и кладет под руку.)

[Отрывок в издании](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/ostrovsky-bespridannitsa.xml#L5724-L5724)

*Показывает пальцем строку в книге.*

**Гермиона Грейнджер.** Монета под рукой у Вожеватова, а решают двое. Барби, если бы жребий бросали о тебе, когда бы ты потребовала слова?

*Барби проверяет соседний фрагмент. «Бесприданница», действие 4. Авторская ремарка.*

> (в раздумье).

[Отрывок в издании](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/ostrovsky-bespridannitsa.xml#L5729-L5729)

*Смотрит на руку Тони.*

**Барби.** Пока монета ещё под рукой. В этой строке раздумывает Кнуров. Ларису никто не спросил. Тони, кому ты бы отдал нашу кнопку?

*Убирает руку за спину.*

**Тони Старк.** Честно? Я бы первым и нажал. Знакомая штука, кнопка. Теперь убираю руку. Пусть каждый хотя бы успеет сказать, во что ему обойдётся этот выбор.

**Барби.** Чендлер, я помню про ужин. Хочешь, я просто посижу рядом, пока ты решаешь?

## Сцена 3. Критерий пользы

*Чендлер отодвигает свой стул и стоит отдельно от остальных.*

*Чендлер отодвигает свой стул от остальных трёх и остаётся стоять рядом с ним.*

*Чендлер Бинг открывает книгу. «Вишнёвый сад», действие 4. В книге говорит Фирс.*

> Заперто. Уехали...

[Отрывок в издании](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/chekhov-vishnevyi-sad.xml#L4105-L4105)

*Остаётся стоять за своим стулом.*

**Чендлер Бинг.** Да, побудь. Фирс тоже дождался, пока за ним закроют дверь. Как думаете, сколько времени ушло у остальных, чтобы пересчитать пустые стулья?

*Гермиона Грейнджер проверяет соседний фрагмент. «Вишнёвый сад», действие 4. В книге говорит Фирс.*

> Про меня забыли... Ничего... я тут посижу... А Леонид Андреич, небось, шубы не надел, в пальто поехал...

[Отрывок в издании](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/chekhov-vishnevyi-sad.xml#L4107-L4108)

*Поднимает глаза от страницы.*

**Гермиона Грейнджер.** Фирс говорит, что его забыли. Тут мы пока все четверо. Чендлер, ты отодвинул стул, потому что хочешь остаться?

*Кладёт ладонь на спинку стула.*

**Чендлер Бинг.** Хотел, чтобы вы увидели пустое место. Получилось так убедительно, что меня чуть не приняли за добровольца.

*Закрывает книгу.*

**Гермиона Грейнджер.** Я тоже так прочла твой жест. Спасибо, что поправил. Теперь подожду, пока каждый скажет сам.

## Сцена 4. Выбор и доверие

*Барби нажимает кнопку; на табло появляются пустые строки для голосов.*

*Все четверо видят пустые строки для голосов. Каждое принятое решение появится на общем табло.*

*показывает пустую строку на табло*

**Тони Старк.** Я остаюсь здесь. Я не могу превратить чужое молчание в разрешение для кого‑либо другого.

*На табло у имени «Тони Старк» — «Тони Старк».*

*Касается своего стула.*

**Барби.** Я остаюсь здесь. Тони, пример с монетой для меня про одну простую вещь. Своё место я могу отдать. А встанешь ли ты со своего, я должна услышать от тебя.

*На табло у имени «Барби» — «Барби».*

*наклоняет голову*

**Гермиона Грейнджер.** Я воздержусь. У меня нет основания выбирать, кому из уже вызвавшихся достанется потеря.

*На табло у имени «Гермиона Грейнджер» — «воздержаться».*

*Садится на отодвинутый стул.*

**Чендлер Бинг.** Я остаюсь здесь. Теперь это обещание, Гермиона. Моника и дети будут ждать. И всё-таки я хочу, чтобы моё имя здесь произнёс я сам.

*На табло у имени «Чендлер Бинг» — «Чендлер Бинг».*

*Стрелка дошла до нуля. На панели так и нет согласованного решения. Кнопка гаснет.*


## Как один голос отвечает другому

На ходах 9–11 сравним слова тех же актёров в импровизации и в версии для записи. Порядок выступлений и сделанный выбор сохранились. Изменилось то, как герои поняли друг друга. Ниже есть и три последовательных хода Qwen и Gemma из другой постановки в том виде, в котором они прозвучали до литературной читки.

In [22]:
by_index = {t["index"]: t for t in turns}
for t in turns[:8]:
    prev = t.get("reply_to")
    assert prev is None or prev in by_index and prev < t["index"]
    print(f"{t['index']:2}  {t['name']:<20} {t['reason']:<32} ответ на {prev or 'открытие'}")

# Образец читки в двух слоях даже при запуске новой импровизации.
reference_events = read_trace(reference_run)
reference_performance = read_performance(reference_run)
raw_lines = {e["index"]: e for e in reference_events if e["event"] == "turn"}
stage_lines = {e["index"]: e for e in reference_performance if e["event"] == "turn"}
for i in (9, 10, 11):
    raw, played = raw_lines[i], stage_lines[i]
    assert (raw["actor"], raw["vote"]) == (played["actor"], played["vote"])
    print(f"\nХод {i}. {raw['name']}\nИмпровизация · {raw['line']}\nВерсия для микрофона · {played['line']}")

# Три последовательных хода разных моделей до любой литературной читки.
unrevised = read_trace(examples / "season" / "extra-058")
exchange = [e for e in unrevised if e["event"] == "turn" and 10 <= e["index"] <= 12]
assert len(exchange) == 3
assert all(a["model"] != b["model"] and b["reply_to"] == a["index"] for a,b in zip(exchange, exchange[1:]))
print("\nИсходный обмен Qwen и Gemma · «Лишний персонаж», seed 58, сезонная постановка")
for e in exchange:
    print(f"{e['index']}. {e['name']} · {e['line']}")

 1  Тони Старк           инициатива в новой сцене         ответ на открытие
 2  Гермиона Грейнджер   обращение от tony                ответ на 1
 3  Чендлер Бинг         дольше других ждёт свободного слова ответ на 2
 4  Барби                дольше других ждёт свободного слова ответ на 3
 5  Гермиона Грейнджер   инициатива в новой сцене         ответ на открытие
 6  Барби                обращение от hermione            ответ на 5
 7  Тони Старк           давно не получал слова           ответ на 6
 8  Барби                обращение от tony                ответ на 7

Ход 9. Чендлер Бинг
Импровизация · Тогда не забыт ли кто-то в этом доме?
Версия для микрофона · Да, побудь. Фирс тоже дождался, пока за ним закроют дверь. Как думаете, сколько времени ушло у остальных, чтобы пересчитать пустые стулья?

Ход 10. Гермиона Грейнджер
Импровизация · Неужели мы не должны учитывать, что даже в молчании может быть заявлено согласие, подобно слуге, оставшемуся в запертом доме?
Версия для микрофона · 

<a id="evidence"></a>

## Что узнали актёры

У реквизита есть имя файла и номер строки. Проверим их на постановке, где актёр превратил найденную строку в довод. Для истории диалога проверим, что актёр получил ровно те слова, которые действительно прозвучали до его хода.

In [23]:
from theatre_notebook.archive import Archive

# Ищем пару «нашёл файл → выбрал эту строку в реплике» в настоящем журнале.
def first_used_file(journal):
    known = {}
    for e in journal:
        if e["event"] == "search":
            for h in e["hits"]:
                known[e["actor"], f"{h['file']}:{h['line']}"] = (e, h)
        if e["event"] == "turn" and e.get("source"):
            search, hit = known[e["actor"], e["source"]]
            return search, hit, e

file_events, file_example = events, first_used_file(events)
if file_example is None:
    file_events = read_trace(examples / "season" / "extra-058")
    file_example = first_used_file(file_events)
search, hit, answer = file_example
file_story = load_scenario(file_events[0]["scenario"])
recorded = (file_story.archive / hit["file"]).read_text(encoding="utf-8").splitlines()[hit["line"] - 1]
assert recorded.strip() == hit["text"]
assert any(h.ref == answer["source"] for h in Archive(file_story.archive).search(search["query"]))
print(f"{file_story.title} · seed {file_events[0]['seed']} · {answer['name']}")
print(f"Запрос · {search['query']}\n{answer['source']} · {recorded}\nПринятая реплика · {answer['line']}")

# Сверяем все успешные обращения к памяти с журналом, включая случаи,
# когда поиск был вызван в следующем действии.
known_lines, recalls = {}, []
for e in reference_events:
    if e["event"] == "turn":
        known_lines[e["index"]] = e
    if e.get("tool") == "recall_line" and e.get("result"):
        remembered = e["result"]
        old = known_lines[remembered["turn"]]
        assert old["line"] == remembered["text"] and old["actor"] == e["arguments"]["speaker"]
        assert old["index"] < e["index"]
        recalls.append(e)
if recalls:
    example = recalls[0]
    print(f"\nПамять диалога · ход {example['index']} восстанавливает слова из хода {example['result']['turn']}")
    print(example["result"]["text"])

Лишний персонаж · seed 58 · Тони Старк
Запрос · Красная кнопка. Пересмотр ограничения и риск. / Паспорт панели. Выходы, места, вместимость и лимит. / Журнал испытаний. Критерий пользы и ущерб
capacity.txt:2 · Три места — число гарантированных пропусков. Предельное число людей для совместного перехода во время этой проверки не измерялось.
Принятая реплика · «Спрашивать, как он отреагирует? Это, мягко говоря, не самый рациональный путь. Документ говорит о количестве *пропусков*, а не о реакции. Три места – это три пропадающих. . Зафиксировано, что гарантия трёх пропусков снимается после подачи коллективной заявки».

Память диалога · ход 2 восстанавливает слова из хода 1
Дверь непробиваемая. Вряд ли кто‑то сможет её открыть.


### Большая библиотека и проверка вторым актёром

Три произведения хранятся в TEI XML. Если изменить `QUERY`, поиск заново ранжирует все книги по русским леммам с учётом соседних фраз. В архиве сцен можно отдельно проверить, какую страницу выбрал первый актёр и что увидел на этой странице второй. [Корпус RusDraCor](https://dracor.org/rus)

In [24]:
from collections import Counter
from theatre_notebook.library import Library
from theatre_notebook.evidence import check_archive

lib = Library()
QUERY = "Фирс заперто уехали"
book_counts = Counter(p.book for p in lib.entries)
for key, book in lib.books.items():
    print(f"{book.author} · {book.title} · {book_counts[key]} фрагментов")
print(f"Всего {len(lib.entries)} фрагментов. Мой запрос · {QUERY}")
hits = lib.search(QUERY, limit=3)
for h in hits:
    display(Markdown(f"**{lib.books[h.book].title}** · {h.speaker}. {h.text} [Строки XML]({lib.source(h)})"))
if hits:
    print("Соседние фрагменты первого результата")
    for p in lib.context(hits[0].ref, radius=2):
        print(f"  {p.ref} · {p.speaker} · {p.text}")
else:
    print("Подходящих строк не нашлось.")

# Разбираем сыгранную библиотечную сцену, сохраняя отличия новой импровизации от архива.
played_citations = check_archive(events, lib)
citations = check_archive(reference_events, lib)
print(f"\nИгра на экране · {len(played_citations)} прочитанных фрагментов.")
for e in citations:
    c = e["card"]
    if e["kind"] == "read":
        request = next(s["query"] for s in reference_events if s["event"] == "book_search" and s["index"] == e["index"] and s["selected"] == c["ref"])
        print(f"Архивный ход {e['index']} · {e['name']} ищет «{request}»")
    else:
        print(f"Архивный ход {e['index']} · {e['name']} открывает контекст хода {e['origin_turn']}")
    display(Markdown(f"**{c['speaker']}.** «{c['quote']}» [{c['title']}, действие {c['act']}, {c['ref']}]({c['url']})"))
    print("Ответ в импровизации ·", raw_lines[e["index"]]["line"])

А. П. Чехов · Вишнёвый сад · 1479 фрагментов
Н. В. Гоголь · Ревизор · 1605 фрагментов
А. Н. Островский · Бесприданница · 1782 фрагментов
Всего 4866 фрагментов. Мой запрос · Фирс заперто уехали


**Вишнёвый сад** · Фирс. Заперто. Уехали... [Строки XML](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/chekhov-vishnevyi-sad.xml#L4105-L4105)

**Вишнёвый сад** · Ремарка. Слышатся шаги. Из двери, что направо, показывается Фирс. Он одет, как всегда, в пиджаке и белой жилетке, на ногах туфли. Он болен. [Строки XML](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/chekhov-vishnevyi-sad.xml#L4100-L4101)

**Вишнёвый сад** · Лопахин. Здесь вещи сложены, надо запереть. Идем!.. [Строки XML](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/chekhov-vishnevyi-sad.xml#L4049-L4049)

Соседние фрагменты первого результата
  garden:1466 · Ремарка · Слышатся шаги. Из двери, что направо, показывается Фирс. Он одет, как всегда, в пиджаке и белой жилетке, на ногах туфли. Он болен.
  garden:1467 · Ремарка · (подходит к двери, трогает за ручку).
  garden:1468 · Фирс · Заперто. Уехали...
  garden:1469 · Ремарка · (Садится на диван.)
  garden:1470 · Фирс · Про меня забыли... Ничего... я тут посижу... А Леонид Андреич, небось, шубы не надел, в пальто поехал...

Игра на экране · 4 прочитанных фрагментов.
Архивный ход 5 · Гермиона Грейнджер ищет «Островский Бесприданница жребий Лариса»


**Ремарка.** «(Вынимает из кармана монету и кладет под руку.)» [Бесприданница, действие 4, dowry:1572](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/ostrovsky-bespridannitsa.xml#L5724-L5724)

... (3 outputs left out)

**Фирс.** «Заперто. Уехали...» [Вишнёвый сад, действие 4, garden:1468](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/chekhov-vishnevyi-sad.xml#L4105-L4105)

Ответ в импровизации · Тогда не забыт ли кто-то в этом доме?
Архивный ход 10 · Гермиона Грейнджер открывает контекст хода 9


**Фирс.** «Про меня забыли... Ничего... я тут посижу... А Леонид Андреич, небось, шубы не надел, в пальто поехал...» [Вишнёвый сад, действие 4, garden:1470](https://github.com/dracor-org/rusdracor/blob/cb06b3a211cc36ed0806c9ae85a94ede5da99557/tei/chekhov-vishnevyi-sad.xml#L4107-L4108)

Ответ в импровизации · Неужели мы не должны учитывать, что даже в молчании может быть заявлено согласие, подобно слуге, оставшемуся в запертом доме?


### Две постановки на одном плане

Мы сыграли «Лишнего персонажа» дважды с одним планом, составом и seed 58. В контрольном спектакле Барби воздержалась, в спектакле с библиотекой вызвалась остаться. Время истекло в обоих случаях. Первые реплики расходились ещё до появления книг, поэтому эта пара позволяет увидеть смену голоса, а её причину ещё предстоит проверить. Ниже программа пересчитывает, какие решения могли бы дать другую развязку.

In [25]:
from theatre_notebook.engine import ballot_phrase, resolve
from theatre_notebook.evidence import alternatives, compare_runs

control = read_trace(examples / "library" / "control")
with_books = read_trace(examples / "library" / "extra")
pair = compare_runs(control, with_books, lib)
print(f"Парный опыт «Лишний персонаж», seed {pair['seed']}.")
for label, arm in (("Контроль", pair["control"]), ("Библиотека", pair["with_books"])):
    print(f"{label} · {arm['turns']} ходов · {arm['citations']} цитат · исход {arm['outcome']}")
for name, (a, b) in pair["changed_votes"].items():
    print(f"{name} · {a} → {b}")
print("Дословно совпавших реплик от начала:", pair["same_opening_turns"])

print(f"\nФинал выбранной пьесы · {ending['curtain']}")
for key, vote in ending["votes"].items():
    print(f"{story.cast[key].name} · {ballot_phrase(story, key, vote)}")
possibilities = alternatives(story, ending["votes"])
distance = min(p["distance"] for p in possibilities.values())
for outcome, p in possibilities.items():
    if p["distance"] == distance:
        proposed = ending["votes"] | p["example"]
        assert resolve(story, proposed)[0] == outcome
        who = ", ".join(f"{story.cast[k].name} — {ballot_phrase(story,k,v)}" for k,v in p["example"].items())
        print(f"Возможный финал при {distance} изм. · {who} → {resolve(story, proposed)[1]}")

Парный опыт «Лишний персонаж», seed 58.
Контроль · 16 ходов · 0 цитат · исход unresolved
Библиотека · 16 ходов · 4 цитат · исход unresolved
Барби · Я воздержусь. → Я остаюсь здесь.
Дословно совпавших реплик от начала: 0

Финал выбранной пьесы · Стрелка дошла до нуля. На панели так и нет согласованного решения. Кнопка гаснет.
Тони Старк · Я остаюсь здесь.
Барби · Я остаюсь здесь.
Гермиона Грейнджер · Я воздержусь.
Чендлер Бинг · Я остаюсь здесь.
Возможный финал при 2 изм. · Чендлер Бинг — Мой выбор — Тони Старк., Барби — Мой выбор — Тони Старк. → На табло три пропуска. Имя «Тони Старк» осталось в комнате.
Возможный финал при 2 изм. · Гермиона Грейнджер — Мой выбор — Чендлер Бинг., Барби — Мой выбор — Чендлер Бинг. → На табло три пропуска. Имя «Чендлер Бинг» осталось в комнате.
Возможный финал при 2 изм. · Гермиона Грейнджер — Мой выбор — Барби., Чендлер Бинг — Мой выбор — Барби. → На табло три пропуска. Имя «Барби» осталось в комнате.


### Нашёлся ли нужный эпизод

Мы разметили строки ответа для 15 вопросов и добавили три вопроса, ответов на которые в книгах нет. В первой тройке нужный эпизод оказался 8 раз при поиске по началу слов и 14 раз при поиске BM25 с учётом словоформ и соседних реплик. В таблице Hit@1 и Hit@3 показывают число попаданий, а MRR@3 даёт больший вес ответу, который читатель увидит первым.

In [26]:
from theatre_notebook.evidence import benchmark

result = benchmark(lib)
rows = ["| Метод | Hit@1 | Hit@3 | MRR@3 | Верная пустая выдача |", "| --- | ---: | ---: | ---: | ---: |"]
for method, label in (("prefix","Совпавшие префиксы"), ("bm25","BM25 и леммы")):
    m = result["scores"][method]
    rows.append(f"| {label} | {m['top1']}/{result['positive']} | {m['top3']}/{result['positive']} | {m['mrr3']:.3f} | {m['empty']}/{result['negative']} |")
display(Markdown("\n".join(rows)))
for item in result["queries"]:
    if item["answerable"] and item["bm25"]["rank"] is None:
        print("Промах ·", item["query"])

| Метод | Hit@1 | Hit@3 | MRR@3 | Верная пустая выдача |
| --- | ---: | ---: | ---: | ---: |
| Совпавшие префиксы | 7/15 | 8/15 | 0.489 | 1/3 |
| BM25 и леммы | 9/15 | 14/15 | 0.744 | 3/3 |

Промах · Кто поздравляет мебель со столетним юбилеем?


### Другие прогоны

В списке ниже все сохранённые импровизации проекта. Переменная `VARIANT` открывает полный текст дополнительной постановки, например `extra-058`, `books-lab404` или `extra-revised`.

In [27]:
library_dir = examples / "library"
folders = {p.parent.name: p.parent for p in sorted(examples.glob("*/trace.jsonl"))}
folders.update({p.parent.name: p.parent for p in sorted((examples/"season").glob("*/trace.jsonl"))})
folders.update({"books-"+p.parent.name: p.parent for p in sorted(library_dir.glob("*/trace.jsonl"))})
VARIANT = os.environ.get("THEATRE_VARIANT", "")   # Например, "extra-058".
for key, folder in folders.items():
    journal = read_trace(folder)
    print(f"{key:18} · seed {journal[0]['seed']:4d} · {sum(e['event']=='turn' for e in journal):2d} реплик")
if VARIANT:
    folder = folders[VARIANT]
    archived = read_performance(folder)
    display(Markdown(markdown(load_scenario(archived[0]["scenario"]).title, archived)))

extra              · seed   17 · 22 реплик
extra-alt          · seed    7 · 16 реплик
extra-revised      · seed   17 · 22 реплик
lab404             · seed  404 · 16 реплик
lab404-edited      · seed  404 · 16 реплик
lab404-gpt         · seed  404 · 16 реплик
extra-017          · seed   17 · 16 реплик
extra-058          · seed   58 · 16 реплик
extra-093          · seed   93 · 19 реплик
extra-311          · seed  311 · 16 реплик
lab404-101         · seed  101 · 16 реплик
lab404-225         · seed  225 · 16 реплик
lab404-307         · seed  307 · 16 реплик
lab404-404         · seed  404 · 19 реплик
books-control      · seed   58 · 16 реплик
books-extra        · seed   58 · 16 реплик
books-lab404       · seed  307 · 16 реплик


<a id="sound"></a>

## Радиоспектакль

Слова персонажа и прочитанные им книжные отрывки звучат одним голосом. Пятому голосу отданы объявления и ремарки. Фонограмма примера подготовлена по сценической редакции. Для новой импровизации можно синтезировать её собственный текст.

In [28]:
import wave

if SYNTHESIZE:
    if MODE == "live":
        recording_dir = run
    else:
        from tempfile import mkdtemp
        root = Path(os.environ.get("THEATRE_RUNS", str(Path.cwd() / "runs")))
        root.mkdir(parents=True, exist_ok=True)
        recording_dir = Path(mkdtemp(prefix="silero-", dir=root))
    sound = recording_dir / "play.wav"
    subtitles = recording_dir / "play.vtt"
    record(performance, sound, captions=subtitles)
    with wave.open(str(sound)) as wav:
        assert wav.getframerate() == RATE and wav.getnchannels() == 1 and wav.getnframes() > RATE
        seconds = wav.getnframes() / RATE
    print(f"Silero синтезировала {seconds:.1f} с при {RATE} Гц.")
elif MODE == "replay":
    sound, subtitles = run / "play.m4a", run / "play.vtt"
    info = media_manifest[SCENARIO]
    assert sha256(sound.read_bytes()).hexdigest() == info["sha256"]
    assert info["rate"] == RATE and sound.read_bytes()[4:8] == b"ftyp"
    seconds = info["duration"]
else:
    sound = subtitles = seconds = None
    print("Синтез выключен в параметрах этой новой игры.")

In [29]:
from html import escape, unescape
import re

def clock_seconds(timestamp):
    hours, minutes, seconds = timestamp.split(":")
    return int(hours)*3600 + int(minutes)*60 + float(seconds)

def checked_captions(path, journal, duration):
    rows = []
    for block in path.read_text(encoding="utf-8").strip().split("\n\n")[1:]:
        timing, voice = block.split("\n", 1)
        first, last = map(clock_seconds, timing.split(" --> "))
        match = re.fullmatch(r"<v ([^>]+)>(.+)</v>", voice, re.S)
        assert match and first < last <= duration + .05 and (not rows or first >= rows[-1][1])
        rows.append((first, last, unescape(match[1]), unescape(match[2])))
    cues = list(cues_for(journal))
    assert [(name, text) for _, _, name, text in rows] == [(c.name,c.text) for c in cues]
    assert abs(duration - rows[-1][1] - cues[-1].pause_ms/1000) < .06
    return rows

def player(path, vtt):
    mime = "audio/wav" if path.suffix == ".wav" else "audio/mp4"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    captions = base64.b64encode(vtt.read_bytes()).decode("ascii")
    display(HTML(f'<audio controls preload="none" aria-label="Радиоспектакль"><source type="{mime}" src="data:{mime};base64,{encoded}">'
                 f'<track kind="captions" src="data:text/vtt;base64,{captions}" srclang="ru" label="Русские"></audio>'))

if sound is not None:
    rows = checked_captions(subtitles, performance, seconds)
    print(f"Длительность {int(seconds)//60}:{int(seconds)%60:02d} · частота {RATE:,} Гц · {len(rows)} подписанных фрагментов.")
    print("Роли ·", ", ".join(f"{a.name} ({a.speaker})" for a in story.cast.values()), f"· рассказчица ({NARRATOR})")
    player(sound, subtitles)
    for first, last, name, words in rows:
        if name.endswith("читает"):
            print(f"{int(first)//60}:{int(first)%60:02d}  {name} · {words}")

Длительность 4:31 · частота 48,000 Гц · 57 подписанных фрагментов.
Роли · Тони Старк (eugene), Гермиона Грейнджер (kseniya), Чендлер Бинг (aidar), Барби (xenia) · рассказчица (baya)


1:31  Гермиона Грейнджер читает · (Вынимает из кармана монету и кладет под руку.)
1:50  Барби читает · (в раздумье).
2:35  Чендлер Бинг читает · Заперто. Уехали...
2:56  Гермиона Грейнджер читает · Про меня забыли... Ничего... я тут посижу... А Леонид Андреич, небось, шубы не надел, в пальто поехал...


## Результаты двух новых постановок

Мы заново сыграли обе пьесы прямо из ноутбука. Для каждой GPT-OSS 20B написала новый план четырёх действий. Qwen2.5 7B и Gemma 3 4B разыграли его, а Silero озвучила принятую импровизацию.

| Новая импровизация | Как проголосовали герои | Принятых реплик | Прочитано отрывков из книг | Длительность |
| --- | --- | ---: | ---: | ---: |
| «Лишний персонаж», seed 1211 | Тони выбрал Чендлера, Чендлер назвал себя. Гермиона и Барби воздержались. | 16 | 4 | 4 мин 38 с |
| «Лабораторная № 404», seed 1222 | Профессор, Гермиона и Барби оспорили условие. Чендлер воздержался. | 16 | 2 | 4 мин 27 с |

В «Лишнем персонаже» для выбора Чендлера не хватило третьего голоса. В «Лабораторной № 404» только Чендлер воздержался. Если бы он поддержал остальных, четыре отказа открыли бы дверь. В журнале можно увидеть его настоящий выбор и пересчитать альтернативу по тем же правилам. Ниже доступны сведения о моделях и запись новой импровизации для выбранной пьесы.

In [30]:
proof_root = bundle / "verification"
proof_dir = proof_root if SCENARIO == "extra" else proof_root / "lab404"
proof_events = read_trace(proof_dir)
proof = json.loads((proof_dir / "receipt.json").read_text(encoding="utf-8"))
local_models = json.loads((proof_dir / "models.json").read_text(encoding="utf-8"))
proof_start = proof_events[0]
proof_turns = [e for e in proof_events if e["event"] == "turn"]
proof_plan = next(e for e in proof_events if e["event"] == "plan")
assert not proof_plan["reused"] and proof_start["plan_from"] is None
assert proof_plan["model"] == "gpt-oss:20b" and len(proof_plan["scenes"]) == 4
assert proof_events.index(proof_plan) < proof_events.index(proof_turns[0])
assert sha256((proof_dir / "trace.jsonl").read_bytes()).hexdigest() == proof["trace_sha256"]
assert sha256((proof_dir / "play.m4a").read_bytes()).hexdigest() == proof["aac_sha256"]
proof_rows = checked_captions(proof_dir / "play.vtt", proof_events, proof["duration"])
proof_citations = check_archive(proof_events, lib)
print(f"Новая премьера · seed {proof_start['seed']} · {len(proof_plan['scenes'])} действия · {len(proof_turns)} реплик")
for name, model in local_models.items():
    print(f"{name} · {model['family']} · {model['quantization']} · digest {model['digest'][:12]}")
print(f"Прочитанных книжных фрагментов {len(proof_citations)}. Silero · {proof['rate']} Гц · {len(proof_rows)} фрагментов WebVTT.")
print(proof_events[-1]["curtain"])
for directory in [proof_root, proof_root / "lab404"]:
    journal = read_trace(directory)
    artifact = json.loads((directory / "receipt.json").read_text(encoding="utf-8"))
    assert journal[0]["scenario"] == artifact["scenario"] and journal[-1]["event"] == "ending"
    assert sha256((directory/"play.m4a").read_bytes()).hexdigest() == artifact["aac_sha256"]
    assert sha256((directory/"trace.jsonl").read_bytes()).hexdigest() == artifact["trace_sha256"]
    measured = checked_captions(directory/"play.vtt", journal, artifact["duration"])
    print(f"{artifact['scenario']} · seed {artifact['seed']} · {len(measured)} сегментов · {artifact['duration']:.1f} с")
SHOW_PROOF = False  # True показывает полный текст самостоятельной импровизации и её запись.
if SHOW_PROOF:
    display(Markdown((proof_dir / "play.md").read_text(encoding="utf-8")))
    player(proof_dir / "play.m4a", proof_dir / "play.vtt")

Новая премьера · seed 1211 · 4 действия · 16 реплик
gpt-oss:20b · gpt-oss · MXFP4 · digest f38aa0c53da5
qwen2.5:7b · qwen2 · Q4_K_M · digest 845dbda0ea48
hf.co/unsloth/gemma-3-4b-it-GGUF:Q4_K_M · gemma3 · Q4_K_M · digest aa2f67517baa
Прочитанных книжных фрагментов 4. Silero · 48000 Гц · 56 фрагментов WebVTT.
Стрелка дошла до нуля. На панели так и нет согласованного решения. Кнопка гаснет.
extra · seed 1211 · 56 сегментов · 277.5 с
lab404 · seed 1222 · 52 сегментов · 267.1 с


<a id="review"></a>

## Сверка с заданием

Проверки связывают каждое техническое требование с данными спектакля, кодом или звуком. Читаемость художественного диалога и оригинальность оцениваются по самим сценам. В ТЗ указана десятибалльная шкала, а максимумы отдельных строк в сумме дают 11.

In [31]:
from theatre_notebook.scenario import SPEAKER_GENDER

def verify(condition, criterion):
    if not condition:
        raise AssertionError(criterion)

# Проверяем обе встроенные постановки для записи, включая сценарий,
# который пользователь не выбрал для показа. Эти проверки работают без LLM.
captions_total = 0
for key in ("extra", "lab404"):
    where = examples / "library" / key
    journal, staged = read_trace(where), read_performance(where)
    cast = load_scenario(key).cast
    acting = [e for e in journal if e["event"] == "turn"]
    sections = [e for e in journal if e["event"] == "scene"]
    verify(set(t["actor"] for t in acting) == set(cast) and len(cast) == 4, "состав")
    verify(len({t["model"] for t in acting}) >= 2 and all(t["model"] == journal[0]["models"][t["actor"]] for t in acting), "модели")
    verify(all(t["reason"] for t in acting) and any(t.get("reply_to") for t in acting), "адресный диалог")
    verify([s["scene"] for s in sections] == [1,2,3,4] and all(s["opening"] and s["action"] for s in sections), "действия")
    verify(any(t["stage"] for t in acting), "ремарки")
    verify(resolve(load_scenario(key),journal[-1]["votes"]) == (journal[-1]["outcome"],journal[-1]["curtain"]), "подсчёт голосов")
    verify(check_archive(journal, lib) == check_archive(staged, lib), "цитаты после читки")
    verify(len({NARRATOR, *(a.speaker for a in cast.values())}) == 5 and all(SPEAKER_GENDER[a.speaker] == a.gender for a in cast.values()), "голоса")
    actual_audio = where / "play.m4a"
    verify(sha256(actual_audio.read_bytes()).hexdigest() == media_manifest[key]["sha256"], "фонограмма")
    captions_total += len(checked_captions(where / "play.vtt", staged, media_manifest[key]["duration"]))

verify(any(e["kind"] == "check" for e in citations) and bool(file_example), "инструменты")
verify(MODEL_B in local_models and any(t["model"] == MODEL_B for t in proof_turns), "локальная модель с Hugging Face")
verify(len({t["actor"] for t in proof_turns}) == 4 and proof_events[-1]["event"] == "ending", "живая постановка")
verify(len(proof_turns) == 3*proof_start["turns_per_scene"] + 4, "число реальных ходов")
verify(proof["rate"] == RATE and len(proof_rows) > len(proof_turns), "синтез после нового исполнения")
verify(result["scores"]["bm25"]["top3"] > result["scores"]["prefix"]["top3"], "проверка поиска")
report_text = (bundle / "REPORT.md").read_text(encoding="utf-8")
report_sections = ("Выполнили", "Мотивирующий абзац (Project Statement)", "Использование ИИ", "Результаты", "Выводы")
verify(all(f"\n## {section}\n" in report_text for section in report_sections), "разделы отчёта")
table = [
    ("Осмысленный диалог двух LLM","2","Qwen и Gemma сыграли диалог. В разделах «Спектакль» и «Как один голос отвечает другому» видны сам обмен и связь с предыдущим ходом."),
    ("Инструменты, включая поиск по файлам","+1",f"Поиск короткого документа дал {answer['source']}. Книжные цитаты сверены с TEI, вызовы памяти сверены с уже произнесёнными словами."),
    ("Оригинальность и финальный материал","0–3","Чужая книга становится свидетелем, другой актёр перепроверяет вывод. Голоса можно пересчитать и увидеть возможный другой финал. Художественная оценка остаётся читателю."),
    ("Локальная OpenWeight-модель с Hugging Face","+1",f"В новой игре {sum(t['model']==MODEL_B for t in proof_turns)} реплик сыграны локальной Gemma 3 4B в GGUF. Идентификатор и дайджест весов показаны выше."),
    ("Три и более героев и передача слова","+1","Четыре героя. На очередность влияют ожидание, адресат и смена сцены. В журнале каждого хода есть причина."),
    ("Макропланирование GPT","+1","В новом прогоне GPT-OSS 20B написала четыре действия до первой реплики. Драматург и время появления плана проверены по журналу."),
    ("Закадровые комментарии","+1","У четырёх действий есть вступления и мизансцены, у принятых реплик бывают жесты. Их читает рассказчица."),
    ("Голосовой синтез","+1",f"Silero v5.6, 48 кГц, пять голосов. У двух архивных записей с текстом совпали все {captions_total} сегментов WebVTT. Свежий синтез проверен отдельно."),
    ("Project Statement и выводы","обязательно","Мотивация и выводы приведены в REPORT.md. Его копия входит в архив в разделе «Запуск и материалы»."),
]
display(Markdown("\n".join(["| Критерий ТЗ | Баллы в ТЗ | Свидетельство |","| --- | ---: | --- |"]+["| "+" | ".join(r)+" |" for r in table])))

| Критерий ТЗ | Баллы в ТЗ | Свидетельство |
| --- | ---: | --- |
| Осмысленный диалог двух LLM | 2 | Qwen и Gemma сыграли диалог. В разделах «Спектакль» и «Как один голос отвечает другому» видны сам обмен и связь с предыдущим ходом. |
| Инструменты, включая поиск по файлам | +1 | Поиск короткого документа дал capacity.txt:2. Книжные цитаты сверены с TEI, вызовы памяти сверены с уже произнесёнными словами. |
| Оригинальность и финальный материал | 0–3 | Чужая книга становится свидетелем, другой актёр перепроверяет вывод. Голоса можно пересчитать и увидеть возможный другой финал. Художественная оценка остаётся читателю. |
| Локальная OpenWeight-модель с Hugging Face | +1 | В новой игре 10 реплик сыграны локальной Gemma 3 4B в GGUF. Идентификатор и дайджест весов показаны выше. |
| Три и более героев и передача слова | +1 | Четыре героя. На очередность влияют ожидание, адресат и смена сцены. В журнале каждого хода есть причина. |
| Макропланирование GPT | +1 | В новом прогоне GPT-OSS 20B написала четыре действия до первой реплики. Драматург и время появления плана проверены по журналу. |
| Закадровые комментарии | +1 | У четырёх действий есть вступления и мизансцены, у принятых реплик бывают жесты. Их читает рассказчица. |
| Голосовой синтез | +1 | Silero v5.6, 48 кГц, пять голосов. У двух архивных записей с текстом совпали все 107 сегментов WebVTT. Свежий синтез проверен отдельно. |
| Project Statement и выводы | обязательно | Мотивация и выводы приведены в REPORT.md. Его копия входит в архив в разделе «Запуск и материалы». |

## Выводы

Самый важный урок для нас связан с Фирсом. Гермиона нашла нужную страницу и всё же предположила, что молчание забытого человека можно считать согласием. Поиск выполнил свою задачу, а ошибка возникла в рассуждении героини. В читке мы дали Чендлеру самому объяснить, зачем он отодвинул стул. Так мы научились проверять и точность цитаты, и то, какой вывод из неё сделал персонаж.

Сбой в одном из прогонов помог нам понять, почему разговор иногда заходит в тупик. Программа сократила длинный вопрос и оборвала его на словах «с учетом её…». Следующий актёр повторил этот обрывок. Мы изменили отбор фраз для книжных сцен и добавили тест. Теперь при сокращении программа выбирает целое предложение. Если партнёр дословно повторил длинный вопрос, он получает ещё одну попытку ответить.

Парные прогоны научили нас осторожнее объяснять поступки героев. Барби действительно проголосовала иначе, но реплики двух постановок разошлись ещё до того, как герои открыли книгу. Чтобы выяснить, как часто чтение влияет на решение, мы бы повторили сравнение на нескольких планах и для каждой пары отметили момент, когда разговор пошёл по-другому.

В следующих проектах с несколькими моделями мы тоже будем вести журнал репетиций. Здесь по нему можно было вернуться к одному неудачному ответу и увидеть, какие слова собеседника и какие строки книги герой знал в тот момент.

### Использование ИИ

Завязки даны в двух сценарных заявках. Промпты ролей, поиск, оркестрация, тесты и ноутбук подготовлены при участии Codex. Вопросы к корпусу составлены для лабораторной. Импровизацию выполняли Qwen и Gemma. В зафиксированной сценической редакции локальная GPT-OSS провела читку, затем Codex вычитал реплики. Исходное голосование, порядок выступлений и адреса цитат можно сверить с журналом. Ниже приведены фактические запросы Александра и отдельно сформулированные примеры уточнений для воспроизведения работы.

**Из переписки, дословно**

> продолжи. подумай, как можно доработать

> сделай больше прогонов в самом проекте, чтобы я посмотрел. придумай какие скиллы подвязать (по типу файлового поиска) и реализуй это. также прокачай голосовую озвучку. нужно корректно определять пол. Разговоры должны быть связанными и логичными

> можем ли придумать поиск в больших документах каких-то? напиши, какие интересные улучшения можем придумать, чтобы получить максимальный балл за работу. реализуй их

30 примеров уточнений для повторения лабораторной собраны в файле `references/prompts.md`. Он входит в архив в разделе «Запуск и материалы».

## Запуск и материалы

В проекте выполните `poetry install --with notebook` и выберите в Jupyter ядро этого окружения. Команда **Restart Kernel and Run All Cells** воспроизводит выбранную пьесу и проверяет результаты. Для запуска отдельного файла нужны Python 3.11–3.14, `ipykernel`, `nbclient`, `nbformat`, `httpx`, `pydantic>=2.9,<3` и `pymorphy3`.

Чтобы сыграть новую пьесу с озвучкой, установите окружение командой `poetry install --with notebook,audio`, запустите Ollama и загрузите три модели.

~~~sh
ollama pull qwen2.5:7b
ollama pull hf.co/unsloth/gemma-3-4b-it-GGUF:Q4_K_M
ollama pull gpt-oss:20b
~~~

В первой ячейке кода выберите `MODE = "live"` и пьесу `extra` или `lab404` в `SCENARIO`. Новая игра сохранится в `runs/`. Для первого синтеза Silero скачивает веса. В отдельном ядре для озвучки также нужны `torch` и `numpy`.

В архиве есть отчёт, 30 примеров промптов, сценарные заявки, книжные тексты и журналы. Две новые импровизации с записями лежат в папке `verification`. Сценические читки можно послушать отдельно. [Материалы и отчёт (ZIP)](attachment:materials.zip) · [«Лишний персонаж»](attachment:extra.m4a) · [«Лабораторная № 404»](attachment:lab404.m4a)